# 1. 분석 배경과 질문

## 분석 목적
서울시 상권분석 데이터를 활용하여 유동인구 대비 매출성과가 높은 상권과 낮은 상권을 선정하고, 두 집단의 직장인구·점포·집객시설 특성을 비교한다.

## 분석 질문
1. 2025년 유동인구 대비 매출성과 상위 5개와 하위 5개 상권은 어디인가?
2. 두 집단의 직장인구, 점포, 면적 및 집객시설은 어떤 차이를 보이는가?
3. 분석 결과에서 특히 눈에 띄는 상권은 어떤 특성을 가지는가?

# 2. 데이터 수집 및 GCS 적재

## 데이터 수집 및 GCS 적재

서울 열린데이터광장 Open API를 통해 다음 데이터를 수집하였다.

- 추정매출
- 점포
- 유동인구
- 직장인구
- 집객시설
- 상권영역
- 자치구 상권변화지표

수집한 CSV 원본은 수정하지 않고 GCS 버킷의 `seoul_commercial/raw/` 경로에 데이터 종류별로 적재하였다.

직장인구·집객시설·상권변화지표 원본에는 여러 연도의 분기가 포함되어 있어, 분석 단계에서 2025년 1분기부터 4분기까지 필터링하였다.

# 3. BigQuery Raw 테이블

## BigQuery Raw 테이블 적재

GCS에 저장한 CSV 7개를 BigQuery의`seoul_commercial_raw` 데이터셋에 각각 별도 테이블로 적재하였다.

각 테이블은 원본 데이터의 구조를 유지했으며, SQL을 통해 행 수, 분석기간, 주요 키와 결측 여부를 확인하였다.

| 테이블 | 행 수 |
|---|---:|
| `sales` | 85,732 |
| `store` | 304,775 |
| `street_population` | 6,595 |
| `worker_population` | 34,386 |
| `visitor_facility` | 33,138 |
| `trade_area_info` | 1,650 |
| `region_rate` | 525 |

# 4. dbt staging/mart 모델링

## dbt 모델링

BigQuery Raw 테이블을 dbt source로 등록하고, staging과 mart 계층으로 변환 로직을 구성하였다.

### Staging

각 원본 테이블과 1:1로 대응하는 staging 모델에서 다음 작업을 수행하였다.

- 컬럼명 영문 표준화
- 데이터 타입 변환
- 2025년 1~4분기 필터링
- 분석에 필요한 컬럼 선택

### Mart

`mart_trade_area_quarterly` 모델에서 매출과 점포를 분기·상권 단위로 집계한 후 유동인구, 직장인구, 집객시설 및 상권정보를 결합하였다.

최종 데이터 grain은 `분기 × 상권`이며, `trade_area_quarter_key`를 고유키로 사용하였다.

dbt test를 통해 고유키의 unique/not_null, 주요 컬럼의 not_null, 분석분기의 accepted_values를 검증하였다.

# 5. Mart 테이블 불러오기 및 데이터 품질 확인

## 5-1 분석 환경 설정

dbt로 생성한 `mart_trade_area_quarterly` 테이블을 Python으로 조회하였다.

이 테이블은 2025년 분기·상권 단위이며, 매출·점포를 먼저 집계한 후 유동인구, 직장인구, 집객시설 및 상권정보를 결합한 결과이다.

분석에 앞서 행 수, 고유키, 분석기간, 결측값 및 주요 수치의 이상 여부를 확인한다.

In [ ]:
from google.cloud import bigquery
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

PROJECT_ID = "bonggeunj"
LOCATION = "asia-northeast3"
MICE_TRADE_AREA_CODE = "3001496"

client = bigquery.Client(
    project=PROJECT_ID,
    location=LOCATION
)

print("BigQuery 연결 완료")

## 5-2. dbt Mart 조회

In [ ]:
query = """
SELECT *
FROM `bonggeunj.seoul_commercial_dbt.mart_trade_area_quarterly`
WHERE quarter_code BETWEEN 20251 AND 20254
"""

df_quarterly = client.query(
    query,
    location=LOCATION
).to_dataframe()

print("데이터 크기:", df_quarterly.shape)
display(df_quarterly.head())

## 5-3. 컬럼과 데이터 타입 확인

In [ ]:
print("전체 컬럼 수:", len(df_quarterly.columns))

column_info = pd.DataFrame({
    "column_name": df_quarterly.columns,
    "dtype": df_quarterly.dtypes.astype(str).values
})

display(column_info)

## 5-4. 분기 확인

In [ ]:
print(
    "포함된 분기:",
    sorted(
        df_quarterly["quarter_code"]
        .unique()
        .tolist()
    )
)

## 5-5. 고유키 중복 확인

In [ ]:
duplicate_key_count = (
    df_quarterly
    .duplicated(
        [
            "quarter_code",
            "trade_area_code"
        ]
    )
    .sum()
)

print(
    "분기+상권 중복 행 수:",
    duplicate_key_count
)

In [ ]:
print(
    "trade_area_quarter_key 중복:",
    df_quarterly[
        "trade_area_quarter_key"
    ].duplicated().sum()
)

print(
    "trade_area_quarter_key 결측:",
    df_quarterly[
        "trade_area_quarter_key"
    ].isna().sum()
)

## 5-6. 주요 컬럼 결측치 확인

In [ ]:
quality_columns = [
    "total_sales",
    "total_sales_count",
    "total_store_count",
    "total_floating_population",
    "total_worker_population",
    "total_visitor_facility_count",
    "trade_area_size_m2"
]

missing_summary = (
    df_quarterly[quality_columns]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missing_summary["missing_rate_pct"] = (
    missing_summary["missing_count"]
    / len(df_quarterly)
    * 100
)

display(missing_summary)

## 5-7. 유동인구 결측 상권 확인

In [ ]:
missing_floating = (
    df_quarterly[
        df_quarterly[
            "total_floating_population"
        ].isna()
    ][
        [
            "quarter_code",
            "trade_area_code",
            "trade_area_name",
            "trade_area_type_name",
            "district_name",
            "total_sales"
        ]
    ]
    .sort_values(
        [
            "trade_area_code",
            "quarter_code"
        ]
    )
)

print(
    "유동인구 결측 행 수:",
    len(missing_floating)
)

print(
    "유동인구 결측 상권 수:",
    missing_floating[
        "trade_area_code"
    ].nunique()
)

display(missing_floating)

## 5-8. 변수별 데이터 보유 분기 수 확인

In [ ]:
coverage_summary = (
    df_quarterly
    .groupby(
        [
            "trade_area_code",
            "trade_area_name"
        ],
        as_index=False
    )
    .agg(
        quarter_count=(
            "quarter_code",
            "nunique"
        ),
        floating_population_quarter_count=(
            "total_floating_population",
            "count"
        ),
        worker_population_quarter_count=(
            "total_worker_population",
            "count"
        ),
        facility_quarter_count=(
            "total_visitor_facility_count",
            "count"
        )
    )
)

incomplete_coverage = coverage_summary[
    (coverage_summary["quarter_count"] < 4)
    | (
        coverage_summary[
            "floating_population_quarter_count"
        ] < 4
    )
    | (
        coverage_summary[
            "worker_population_quarter_count"
        ] < 4
    )
    | (
        coverage_summary[
            "facility_quarter_count"
        ] < 4
    )
].copy()

print(
    "하나 이상의 자료가 불완전한 상권 수:",
    len(incomplete_coverage)
)

display(
    incomplete_coverage.sort_values(
        [
            "floating_population_quarter_count",
            "worker_population_quarter_count",
            "facility_quarter_count"
        ]
    )
)

## 5-9. 0 이하 값 확인

In [ ]:
invalid_value_summary = pd.Series({
    "sales_zero_or_negative": (
        df_quarterly["total_sales"] <= 0
    ).sum(),

    "sales_count_zero_or_negative": (
        df_quarterly["total_sales_count"] <= 0
    ).sum(),

    "store_count_zero_or_negative": (
        df_quarterly["total_store_count"] <= 0
    ).sum(),

    "floating_population_zero_or_negative": (
        df_quarterly[
            "total_floating_population"
        ] <= 0
    ).sum(),

    "trade_area_size_zero_or_negative": (
        df_quarterly["trade_area_size_m2"] <= 0
    ).sum()
}).to_frame("row_count")

display(invalid_value_summary)

## 5-10. 주요 수치 요약

In [ ]:
numeric_summary_columns = [
    "total_sales",
    "total_sales_count",
    "total_store_count",
    "total_floating_population",
    "total_worker_population",
    "total_visitor_facility_count",
    "trade_area_size_m2"
]

display(
    df_quarterly[
        numeric_summary_columns
    ]
    .describe()
)

## 5-11. 자동 검증

In [ ]:
assert duplicate_key_count == 0, (
    "분기+상권 중복이 존재합니다."
)

assert set(
    df_quarterly["quarter_code"].unique()
) == {
    20251,
    20252,
    20253,
    20254
}, "2025년 외 분기가 포함되어 있습니다."

assert (
    df_quarterly[
        "trade_area_quarter_key"
    ].isna().sum()
    == 0
), "고유키 결측이 존재합니다."

assert len(df_quarterly) > 0, "조회 결과가 비어 있습니다."

print("기본 데이터 품질 검증 통과")

## 데이터 품질 확인 결과

- dbt Mart 테이블은 총 6,276행이며, 한 행은 `분기 × 상권`을 의미한다.
- `quarter_code + trade_area_code` 결합키의 중복은 존재하지 않았다.
- 분석기간은 2025년 1분기부터 4분기까지로 확인되었다.
- 매출, 매출건수, 점포 및 상권 면적에는 주요 결측이 없었다.
- 유동인구, 직장인구 및 집객시설은 일부 상권·분기에 결측이 존재하였다.
- 유동인구는 매출성과 지표의 분모이므로 매출·매출건수·유동인구가 4개 분기에 모두 존재하는 상권만 연간 성과 순위에 사용한다.
- 직장인구와 집객시설은 4개 분기가 모두 존재하는 상권의 값만 비교분석에 사용하며, 일부 분기만 존재하는 값은 0으로 대체하지 않는다.

# 6. 2025년 연간 상권 데이터 생성

## 6-1 분석 방법

dbt Mart는 `분기 × 상권` 단위이므로 상권별 연간 성과를 비교하기 위해 2025년 4개 분기 데이터를 상권 단위로 집계하였다.

- 매출·매출건수·유동인구: 4개 분기 합계
- 점포·직장인구·집객시설: 분기 평균
- 상권 면적과 행정구역 정보: 고정값
- 상권변화지표: 2025년 가장 최근 분기 값

결측값만 있는 변수가 합계 0으로 변환되지 않도록 `min_count=1`을 적용하였다.

## 6-2 분기 순서 정렬

In [ ]:
df_quarterly = (
    df_quarterly
    .sort_values(
        [
            "trade_area_code",
            "quarter_code"
        ]
    )
    .reset_index(drop=True)
)

## 6-3 결측 보존 합계 함수

In [ ]:
def sum_with_nan(series):
    """
    값이 하나 이상 있으면 합계를 계산하고,
    모든 값이 결측이면 NaN을 유지한다.
    """
    return series.sum(min_count=1)

## 6-4 연간 상권 단위 집계

In [ ]:
df_annual = (
    df_quarterly
    .groupby(
        "trade_area_code",
        as_index=False
    )
    .agg(
        # ---------------------------------
        # 상권 기본정보
        # ---------------------------------
        trade_area_name=(
            "trade_area_name",
            "first"
        ),
        trade_area_type_code=(
            "trade_area_type_code",
            "first"
        ),
        trade_area_type_name=(
            "trade_area_type_name",
            "first"
        ),

        district_code=(
            "district_code",
            "first"
        ),
        district_name=(
            "district_name",
            "first"
        ),

        administrative_dong_code=(
            "administrative_dong_code",
            "first"
        ),
        administrative_dong_name=(
            "administrative_dong_name",
            "first"
        ),

        x_coordinate=(
            "x_coordinate",
            "first"
        ),
        y_coordinate=(
            "y_coordinate",
            "first"
        ),

        trade_area_size_m2=(
            "trade_area_size_m2",
            "first"
        ),

        # ---------------------------------
        # 데이터 제공 분기 수
        # ---------------------------------
        quarter_count=(
            "quarter_code",
            "nunique"
        ),

        sales_quarter_count=(
            "total_sales",
            "count"
        ),

        sales_count_quarter_count=(
            "total_sales_count",
            "count"
        ),

        floating_population_quarter_count=(
            "total_floating_population",
            "count"
        ),

        worker_population_quarter_count=(
            "total_worker_population",
            "count"
        ),

        facility_quarter_count=(
            "total_visitor_facility_count",
            "count"
        ),

        # ---------------------------------
        # 연간 매출 합계
        # ---------------------------------
        total_sales=(
            "total_sales",
            sum_with_nan
        ),

        total_sales_count=(
            "total_sales_count",
            sum_with_nan
        ),

        weekday_sales=(
            "weekday_sales",
            sum_with_nan
        ),

        weekend_sales=(
            "weekend_sales",
            sum_with_nan
        ),

        sales_00_06=(
            "sales_00_06",
            sum_with_nan
        ),

        sales_06_11=(
            "sales_06_11",
            sum_with_nan
        ),

        sales_11_14=(
            "sales_11_14",
            sum_with_nan
        ),

        sales_14_17=(
            "sales_14_17",
            sum_with_nan
        ),

        sales_17_21=(
            "sales_17_21",
            sum_with_nan
        ),

        sales_21_24=(
            "sales_21_24",
            sum_with_nan
        ),

        # ---------------------------------
        # 유동인구
        # ---------------------------------
        total_floating_population=(
            "total_floating_population",
            sum_with_nan
        ),

        avg_quarterly_floating_population=(
            "total_floating_population",
            "mean"
        ),

        weekday_floating_population=(
            "weekday_floating_population",
            sum_with_nan
        ),

        weekend_floating_population=(
            "weekend_floating_population",
            sum_with_nan
        ),

        flow_00_06=(
            "flow_00_06",
            sum_with_nan
        ),

        flow_06_11=(
            "flow_06_11",
            sum_with_nan
        ),

        flow_11_14=(
            "flow_11_14",
            sum_with_nan
        ),

        flow_14_17=(
            "flow_14_17",
            sum_with_nan
        ),

        flow_17_21=(
            "flow_17_21",
            sum_with_nan
        ),

        flow_21_24=(
            "flow_21_24",
            sum_with_nan
        ),

        # ---------------------------------
        # 점포: 분기 평균
        # ---------------------------------
        avg_store_count=(
            "total_store_count",
            "mean"
        ),

        avg_non_franchise_store_count=(
            "non_franchise_store_count",
            "mean"
        ),

        avg_franchise_store_count=(
            "franchise_store_count",
            "mean"
        ),

        avg_open_store_count=(
            "open_store_count",
            "mean"
        ),

        avg_close_store_count=(
            "close_store_count",
            "mean"
        ),

        avg_sales_industry_count=(
            "sales_industry_count",
            "mean"
        ),

        # ---------------------------------
        # 직장인구: 분기 평균
        # ---------------------------------
        avg_worker_population=(
            "total_worker_population",
            "mean"
        ),

        avg_male_worker_population=(
            "male_worker_population",
            "mean"
        ),

        avg_female_worker_population=(
            "female_worker_population",
            "mean"
        ),

        # ---------------------------------
        # 집객시설: 분기 평균
        # ---------------------------------
        avg_visitor_facility_count=(
            "total_visitor_facility_count",
            "mean"
        ),

        avg_transport_facility_count=(
            "transport_facility_count",
            "mean"
        ),

        avg_shopping_culture_facility_count=(
            "shopping_culture_facility_count",
            "mean"
        ),

        avg_medical_facility_count=(
            "medical_facility_count",
            "mean"
        ),

        avg_education_facility_count=(
            "education_facility_count",
            "mean"
        ),

        avg_public_finance_facility_count=(
            "public_finance_facility_count",
            "mean"
        ),

        avg_accommodation_facility_count=(
            "accommodation_facility_count",
            "mean"
        ),

        # ---------------------------------
        # 가장 최근 분기의 상권변화지표
        # ---------------------------------
        trade_area_change_index=(
            "trade_area_change_index",
            "last"
        ),

        trade_area_change_name=(
            "trade_area_change_name",
            "last"
        ),

        operating_month_average=(
            "operating_month_average",
            "last"
        ),

        closing_month_average=(
            "closing_month_average",
            "last"
        ),

        seoul_operating_month_average=(
            "seoul_operating_month_average",
            "last"
        ),

        seoul_closing_month_average=(
            "seoul_closing_month_average",
            "last"
        )
    )
)

print("연간 데이터 크기:", df_annual.shape)
display(df_annual.head())

## 6-5. 연간 데이터 키 확인

In [ ]:
print(
    "전체 상권 수:",
    len(df_annual)
)

print(
    "고유 상권코드 수:",
    df_annual["trade_area_code"].nunique()
)

print(
    "상권코드 중복:",
    df_annual[
        "trade_area_code"
    ].duplicated().sum()
)

## 6-6. 분기 보유 현황 확인

In [ ]:
quarter_coverage_summary = (
    df_annual[
        "quarter_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("quarter_count")
    .reset_index(name="trade_area_count")
)

display(quarter_coverage_summary)

In [ ]:
floating_coverage_summary = (
    df_annual[
        "floating_population_quarter_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "floating_population_quarter_count"
    )
    .reset_index(
        name="trade_area_count"
    )
)

display(floating_coverage_summary)

## 6-7. 연간 결측치 확인

In [ ]:
annual_quality_columns = [
    "total_sales",
    "total_sales_count",
    "total_floating_population",
    "avg_store_count",
    "avg_worker_population",
    "avg_visitor_facility_count",
    "trade_area_size_m2"
]

annual_missing_summary = (
    df_annual[
        annual_quality_columns
    ]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

annual_missing_summary["missing_rate_pct"] = (
    annual_missing_summary["missing_count"]
    / len(df_annual)
    * 100
)

display(annual_missing_summary)

## 6-8. 점포 수 계산 검증

In [ ]:
store_difference = (
    df_annual["avg_store_count"]
    - (
        df_annual[
            "avg_non_franchise_store_count"
        ]
        + df_annual[
            "avg_franchise_store_count"
        ]
    )
)

print(
    "점포 합계 불일치 상권 수:",
    (store_difference.abs() > 0.001).sum()
)

## 6-9. 자동 검증

In [ ]:
assert (
    df_annual[
        "trade_area_code"
    ].duplicated().sum()
    == 0
), "연간 데이터에 상권코드 중복이 있습니다."

assert (
    df_annual["trade_area_code"].nunique()
    == len(df_annual)
), "한 상권당 한 행이 아닙니다."

assert (
    store_difference.abs() > 0.001
).sum() == 0, "점포 수 합계가 일치하지 않습니다."

print("연간 상권 데이터 검증 통과")

## 연간 상권 데이터 생성 결과

분기·상권 단위의 dbt Mart 데이터를 상권코드 기준으로 집계하여 2025년 연간 상권 데이터를 생성하였다.

매출·매출건수·유동인구는 4개 분기 합계로 계산했으며, 점포·직장인구·집객시설은 분기별 평균을 사용하였다.

상권코드의 중복 여부를 확인한 결과 한 상권당 한 행으로 구성되었으며, 전체 점포 수는 비프랜차이즈 점포 수와 프랜차이즈 점포 수의 합과 일치하는 것으로 확인되었다.

다음 단계에서는 유동인구 대비 매출, 유동인구 대비 거래건수, 객단가, 점포당 매출, 직장인구 집중도 및 면적 보정지표를 계산한다.

이후 분석은 유동인구 대비 매출성과를 비교하는 것이 목적이므로, 매출·매출건수·유동인구가 2025년 4개 분기에 모두 존재하고 각 합계가 0보다 큰 상권만 사용한다. 이 조건을 충족한 분석 대상은 1,558개 상권이다.

# 7. 분석용 파생지표 계산

## 7-1. 지표 정의

상권별 총매출과 유동인구의 절대 규모만으로는 상권 간 성과를 직접 비교하기 어렵기 때문에 매출·거래·점포·직장인구·집객시설·면적을 고려한 파생지표를 계산하였다.

주요 성과지표는 `총매출 ÷ 총유동인구`로 정의한 유동인구 대비 매출이다. 여기서 유동인구는 고유 방문자 수가 아닌 추정 유동량이며, 추정매출과 유동인구의 포착 범위도 서로 다를 수 있다. 따라서 이 지표는 방문자 1인당 실제 소비액이나 구매 전환율이 아니라, 데이터에서 관측된 상권 간 상대적 매출성과로 해석한다.

`총매출건수 ÷ 총유동인구` 역시 실제 구매 전환율이 아니라 측정 유동인구 대비 거래 발생 강도를 나타내는 보조지표로 사용한다.

### 분석 대상 선정

유동인구 대비 매출성과를 계산하려면 2025년 전체 매출과 유동인구가 동일한 4개 분기에 모두 존재해야 한다. 따라서 다음 조건을 충족하는 상권만 분석 대상으로 선정하였다.

- 매출 데이터가 2025년 4개 분기 모두 존재
- 매출건수 데이터가 2025년 4개 분기 모두 존재
- 유동인구 데이터가 2025년 4개 분기 모두 존재
- 총매출과 총매출건수가 0보다 큼
- 총유동인구가 0보다 큼

유동인구 결측치는 실제 유동인구 0명과 구분하기 위해 대체하지 않고 제외하였다. 원본 연간 데이터는 `df_annual`로 보존하고, 분석용 데이터는 `df_analysis`로 별도 생성하였다.

In [ ]:
# 분석 전 전체 상권 수
total_trade_areas = len(df_annual)

# 분석 조건별 제외 현황
filter_summary = pd.Series({
    "전체 상권": total_trade_areas,

    "매출 4개 분기 미만": (
        df_annual["quarter_count"] < 4
    ).sum(),

    "매출값 4개 분기 미만": (
        df_annual["sales_quarter_count"] < 4
    ).sum(),

    "매출건수값 4개 분기 미만": (
        df_annual["sales_count_quarter_count"] < 4
    ).sum(),

    "유동인구 4개 분기 미만": (
        df_annual[
            "floating_population_quarter_count"
        ] < 4
    ).sum(),

    "총매출 0 이하 또는 결측": (
        df_annual["total_sales"].isna()
        | (df_annual["total_sales"] <= 0)
    ).sum(),

    "매출건수 0 이하 또는 결측": (
        df_annual["total_sales_count"].isna()
        | (df_annual["total_sales_count"] <= 0)
    ).sum(),

    "유동인구 0 이하 또는 결측": (
        df_annual[
            "total_floating_population"
        ].isna()
        | (
            df_annual[
                "total_floating_population"
            ] <= 0
        )
    ).sum()
}).to_frame("trade_area_count")

display(filter_summary)

In [ ]:
df_analysis = df_annual[
    (df_annual["quarter_count"] == 4)
    & (df_annual["sales_quarter_count"] == 4)
    & (df_annual["sales_count_quarter_count"] == 4)
    & (
        df_annual[
            "floating_population_quarter_count"
        ] == 4
    )
    & df_annual["total_sales"].notna()
    & (df_annual["total_sales"] > 0)
    & df_annual["total_sales_count"].notna()
    & (df_annual["total_sales_count"] > 0)
    & df_annual[
        "total_floating_population"
    ].notna()
    & (
        df_annual[
            "total_floating_population"
        ] > 0
    )
].copy()

## 7-2. 나눗셈 함수

In [ ]:
def safe_divide(numerator, denominator):
    """
    분모가 0이면 NaN으로 처리하는 나눗셈 함수
    """
    denominator = denominator.replace(
        0,
        np.nan
    )

    return numerator / denominator

## 7-3. 핵심 매출성과 지표

In [ ]:
# 유동인구 대비 매출
df_analysis["sales_per_flow"] = safe_divide(
    df_analysis["total_sales"],
    df_analysis["total_floating_population"]
)

# 유동인구 대비 거래건수
df_analysis["transactions_per_flow"] = safe_divide(
    df_analysis["total_sales_count"],
    df_analysis["total_floating_population"]
)

# 거래 1건당 평균 매출액
df_analysis["average_transaction_value"] = safe_divide(
    df_analysis["total_sales"],
    df_analysis["total_sales_count"]
)

# 평균 점포당 연간 매출
df_analysis["sales_per_store"] = safe_divide(
    df_analysis["total_sales"],
    df_analysis["avg_store_count"]
)

## 7-4 직장인구 집중도

In [ ]:
df_analysis["worker_concentration"] = safe_divide(
    df_analysis["avg_worker_population"],
    df_analysis[
        "avg_quarterly_floating_population"
    ]
)

## 7-5. 주말 및 시간대 비중

In [ ]:
# 주말 매출 비중
df_analysis["weekend_sales_share"] = safe_divide(
    df_analysis["weekend_sales"],
    df_analysis["total_sales"]
)

# 주말 유동인구 비중
df_analysis["weekend_flow_share"] = safe_divide(
    df_analysis["weekend_floating_population"],
    df_analysis["total_floating_population"]
)

# 17시 이후 매출
df_analysis["evening_sales"] = (
    df_analysis["sales_17_21"]
    + df_analysis["sales_21_24"]
)

# 17시 이후 유동인구
df_analysis["evening_floating_population"] = (
    df_analysis["flow_17_21"]
    + df_analysis["flow_21_24"]
)

# 저녁·야간 매출 비중
df_analysis["evening_sales_share"] = safe_divide(
    df_analysis["evening_sales"],
    df_analysis["total_sales"]
)

# 저녁·야간 유동인구 비중
df_analysis["evening_flow_share"] = safe_divide(
    df_analysis["evening_floating_population"],
    df_analysis["total_floating_population"]
)

## 7-6. 점포 경쟁 관련 지표

In [ ]:
# 유동인구 1만 명당 점포 수
df_analysis["stores_per_10k_flow"] = (
    safe_divide(
        df_analysis["avg_store_count"],
        df_analysis[
            "avg_quarterly_floating_population"
        ]
    )
    * 10_000
)

# 프랜차이즈 점포 비중
df_analysis["franchise_store_share"] = safe_divide(
    df_analysis[
        "avg_franchise_store_count"
    ],
    df_analysis["avg_store_count"]
)

# 개업 점포 비중
df_analysis["open_store_share"] = safe_divide(
    df_analysis["avg_open_store_count"],
    df_analysis["avg_store_count"]
)

# 폐업 점포 비중
df_analysis["close_store_share"] = safe_divide(
    df_analysis["avg_close_store_count"],
    df_analysis["avg_store_count"]
)

# 분기 평균 순점포 변화
df_analysis["avg_net_store_change"] = (
    df_analysis["avg_open_store_count"]
    - df_analysis["avg_close_store_count"]
)

## 7-7. 상권 면적을 ㎢로 변환

In [ ]:
df_analysis["trade_area_size_km2"] = (
    df_analysis["trade_area_size_m2"]
    / 1_000_000
)

## 7-8. 면적 보정지표

In [ ]:
# 1㎢당 연간 매출
df_analysis["sales_per_km2"] = safe_divide(
    df_analysis["total_sales"],
    df_analysis["trade_area_size_km2"]
)

# 1㎢당 연간 유동인구
df_analysis["flow_per_km2"] = safe_divide(
    df_analysis["total_floating_population"],
    df_analysis["trade_area_size_km2"]
)

# 1㎢당 평균 점포 수
df_analysis["stores_per_km2"] = safe_divide(
    df_analysis["avg_store_count"],
    df_analysis["trade_area_size_km2"]
)

# 1㎢당 평균 직장인구
df_analysis["workers_per_km2"] = safe_divide(
    df_analysis["avg_worker_population"],
    df_analysis["trade_area_size_km2"]
)

# 1㎢당 평균 집객시설 수
df_analysis["facilities_per_km2"] = safe_divide(
    df_analysis[
        "avg_visitor_facility_count"
    ],
    df_analysis["trade_area_size_km2"]
)

## 7-9. 시설 유형별 밀도

In [ ]:
# 교통시설 밀도
df_analysis[
    "transport_facilities_per_km2"
] = safe_divide(
    df_analysis[
        "avg_transport_facility_count"
    ],
    df_analysis["trade_area_size_km2"]
)

# 쇼핑·문화시설 밀도
df_analysis[
    "shopping_culture_facilities_per_km2"
] = safe_divide(
    df_analysis[
        "avg_shopping_culture_facility_count"
    ],
    df_analysis["trade_area_size_km2"]
)

# 의료시설 밀도
df_analysis[
    "medical_facilities_per_km2"
] = safe_divide(
    df_analysis[
        "avg_medical_facility_count"
    ],
    df_analysis["trade_area_size_km2"]
)

# 교육시설 밀도
df_analysis[
    "education_facilities_per_km2"
] = safe_divide(
    df_analysis[
        "avg_education_facility_count"
    ],
    df_analysis["trade_area_size_km2"]
)

# 공공·금융시설 밀도
df_analysis[
    "public_finance_facilities_per_km2"
] = safe_divide(
    df_analysis[
        "avg_public_finance_facility_count"
    ],
    df_analysis["trade_area_size_km2"]
)

# 숙박시설 밀도
df_analysis[
    "accommodation_facilities_per_km2"
] = safe_divide(
    df_analysis[
        "avg_accommodation_facility_count"
    ],
    df_analysis["trade_area_size_km2"]
)

## 7-10. 유동인구 대비 집객시설

In [ ]:
df_analysis["facilities_per_10k_flow"] = (
    safe_divide(
        df_analysis[
            "avg_visitor_facility_count"
        ],
        df_analysis[
            "avg_quarterly_floating_population"
        ]
    )
    * 10_000
)

## 7-11. 무한대 값 정리

In [ ]:
df_analysis = df_analysis.replace(
    [np.inf, -np.inf],
    np.nan
)

## 7-12. 직장인구·집객시설 분석 데이터 분리

In [ ]:
# 일부 분기만 존재하는 직장인구·집객시설 값은 비교에 사용하지 않는다.
worker_columns = [
    "avg_worker_population",
    "avg_male_worker_population",
    "avg_female_worker_population",
    "worker_concentration",
    "workers_per_km2",
]

facility_columns = [
    "avg_visitor_facility_count",
    "avg_transport_facility_count",
    "avg_shopping_culture_facility_count",
    "avg_medical_facility_count",
    "avg_education_facility_count",
    "avg_public_finance_facility_count",
    "avg_accommodation_facility_count",
    "facilities_per_km2",
    "facilities_per_10k_flow",
    "transport_facilities_per_km2",
    "shopping_culture_facilities_per_km2",
    "medical_facilities_per_km2",
    "education_facilities_per_km2",
    "public_finance_facilities_per_km2",
    "accommodation_facilities_per_km2",
]

df_analysis.loc[
    df_analysis["worker_population_quarter_count"] < 4,
    worker_columns,
] = np.nan

df_analysis.loc[
    df_analysis["facility_quarter_count"] < 4,
    facility_columns,
] = np.nan

worker_analysis = df_analysis[
    df_analysis["worker_population_quarter_count"] == 4
].copy()

facility_analysis = df_analysis[
    df_analysis["facility_quarter_count"] == 4
].copy()

print("직장인구 비교 가능 상권 수:", len(worker_analysis))
print("집객시설 비교 가능 상권 수:", len(facility_analysis))

## 7-13. 파생지표 확인

In [ ]:
print(df_analysis.columns)

In [ ]:
derived_columns = [
    "trade_area_code",
    "trade_area_name",
    "trade_area_type_name",
    "district_name",

    "total_sales",
    "total_floating_population",
    "sales_per_flow",

    "transactions_per_flow",
    "average_transaction_value",

    "avg_store_count",
    "sales_per_store",
    "stores_per_10k_flow",

    "avg_worker_population",
    "worker_concentration",

    "weekend_sales_share",
    "weekend_flow_share",
    "evening_sales_share",
    "evening_flow_share",

    "trade_area_size_km2",
    "sales_per_km2",
    "flow_per_km2",
    "stores_per_km2",

    "avg_visitor_facility_count",
    "facilities_per_km2"
]

display(
    df_analysis[
        derived_columns
    ].head()
)

## 7-14. 주요 파생지표 요약통계

In [ ]:
metric_columns = [
    "sales_per_flow",
    "transactions_per_flow",
    "average_transaction_value",
    "sales_per_store",
    "worker_concentration",
    "weekend_sales_share",
    "weekend_flow_share",
    "evening_sales_share",
    "evening_flow_share",
    "stores_per_10k_flow",
    "franchise_store_share",
    "sales_per_km2",
    "flow_per_km2",
    "stores_per_km2",
    "facilities_per_km2",
    "facilities_per_10k_flow"
]

metric_summary = (
    df_analysis[
        metric_columns
    ]
    .describe()
    .T
)

display(metric_summary)

In [ ]:
outlier_columns = [
    "trade_area_code",
    "trade_area_name",
    "trade_area_type_name",
    "district_name",
    "total_sales",
    "total_floating_population",
    "sales_per_flow",
    "transactions_per_flow",
    "average_transaction_value",
    "avg_store_count",
    "avg_sales_industry_count",
    "stores_per_10k_flow"
]

display(
    df_analysis
    .nlargest(
        20,
        "sales_per_flow"
    )[outlier_columns]
)

In [ ]:
display(
    df_analysis
    .nsmallest(
        20,
        "total_floating_population"
    )[outlier_columns]
)

## 7-15. 파생지표 결측 확인

In [ ]:
metric_missing = (
    df_analysis[
        metric_columns
    ]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

metric_missing["missing_rate_pct"] = (
    metric_missing["missing_count"]
    / len(df_analysis)
    * 100
)

display(metric_missing)

## 7-16. 매출성과 분해식 검증

유동인구 대비 매출
=
유동인구 대비 거래건수
× 객단가

In [ ]:
reconstructed_sales_per_flow = (
    df_analysis["transactions_per_flow"]
    * df_analysis["average_transaction_value"]
)

decomposition_difference = (
    df_analysis["sales_per_flow"]
    - reconstructed_sales_per_flow
).abs()

print(
    "분해식 최대 오차:",
    decomposition_difference.max()
)

print(
    "오차가 0.001보다 큰 상권 수:",
    (
        decomposition_difference
        > 0.001
    ).sum()
)

## 7-17. 기본 자동 검증

In [ ]:
assert (df_analysis["sales_quarter_count"] == 4).all(), \
    "매출이 4개 분기 미만인 상권이 포함됐습니다."

assert (df_analysis["sales_count_quarter_count"] == 4).all(), \
    "매출건수가 4개 분기 미만인 상권이 포함됐습니다."

assert (
    df_analysis[
        "floating_population_quarter_count"
    ] == 4
).all(), "유동인구가 4개 분기 미만인 상권이 포함됐습니다."

assert (
    df_analysis[
        "total_floating_population"
    ] > 0
).all(), "유동인구가 0 이하인 상권이 포함됐습니다."

assert (
    df_analysis["total_sales"] > 0
).all(), "총매출이 0 이하인 상권이 포함됐습니다."

assert (
    df_analysis["total_sales_count"] > 0
).all(), "매출건수가 0 이하인 상권이 포함됐습니다."

assert (
    decomposition_difference.dropna()
    < 0.001
).all(), "매출성과 분해식이 일치하지 않습니다."

print("분석용 파생지표 검증 통과")

# 8. 성과 순위 진단과 상권 유형별 분류

## 8-1. 순위와 유동인구 백분위 추가

In [ ]:
df_analysis["sales_per_flow_rank"] = (
    df_analysis["sales_per_flow"]
    .rank(method="min", ascending=False)
    .astype("Int64")
)

df_analysis["flow_percentile"] = (
    df_analysis["total_floating_population"]
    .rank(pct=True)
    .mul(100)
)

rank_columns = [
    "sales_per_flow_rank",
    "trade_area_code",
    "trade_area_name",
    "trade_area_type_name",
    "district_name",
    "total_sales",
    "total_floating_population",
    "flow_percentile",
    "sales_per_flow",
    "transactions_per_flow",
    "average_transaction_value"
]

display(
    df_analysis
    .sort_values("sales_per_flow_rank")
    [rank_columns]
    .head(20)
)

## 8-2. 작은 분모와 상위 성과의 중복 확인

In [ ]:
top_20_codes = set(
    df_analysis.nlargest(20, "sales_per_flow")["trade_area_code"]
)

low_flow_20_codes = set(
    df_analysis.nsmallest(
        20,
        "total_floating_population"
    )["trade_area_code"]
)

overlap_codes = top_20_codes & low_flow_20_codes

overlap_result = (
    df_analysis[
        df_analysis["trade_area_code"].isin(overlap_codes)
    ]
    .sort_values("sales_per_flow", ascending=False)
)

print("매출성과 상위 20개 상권 수:", len(top_20_codes))
print("유동인구 하위 20개 상권 수:", len(low_flow_20_codes))
print("두 조건 중복 상권 수:", len(overlap_codes))

display(overlap_result[rank_columns])

## 8-3. 유동인구 기준 확인

In [ ]:
flow_quantiles = (
    df_analysis["total_floating_population"]
    .quantile([0, 0.01, 0.05, 0.10, 0.25, 0.50])
    .rename_axis("quantile")
    .reset_index(name="total_floating_population")
)

display(flow_quantiles)

## 8-4. 주 분석용 데이터 생성

주 분석에서는 유동인구 대비 매출성과의 상·하위 상권을 확인하는 동시에, 극단적인 값이 작은 유동인구나 제한적인 상업 기반에서 비롯됐는지 판단할 수 있도록 별도의 진단 플래그를 추가한다.

In [ ]:
df_ranking = df_analysis.copy()

print("최종 순위 분석 상권 수:", len(df_ranking))

## 8-5. 상권 유형 내부 순위와 백분위 계산

In [ ]:
# 전체 순위: 1위가 가장 높은 성과
df_ranking["sales_per_flow_rank"] = (
    df_ranking["sales_per_flow"]
    .rank(method="min", ascending=False)
    .astype("Int64")
)

# 백분위: 100에 가까울수록 값이 큼
df_ranking["sales_per_flow_percentile"] = (
    df_ranking["sales_per_flow"]
    .rank(pct=True)
    .mul(100)
)

df_ranking["sales_percentile"] = (
    df_ranking["total_sales"]
    .rank(pct=True)
    .mul(100)
)

df_ranking["flow_percentile"] = (
    df_ranking["total_floating_population"]
    .rank(pct=True)
    .mul(100)
)

display(
    df_ranking[
        [
            "trade_area_name",
            "total_sales",
            "sales_percentile",
            "total_floating_population",
            "flow_percentile",
            "sales_per_flow",
            "sales_per_flow_rank",
            "sales_per_flow_percentile"
        ]
    ].head()
)

## 8-6. 해석용 플래그 추가

In [ ]:
# 성과 상·하위 5%
df_ranking["high_sales_per_flow_flag"] = (
    df_ranking["sales_per_flow_percentile"] >= 95
)

df_ranking["low_sales_per_flow_flag"] = (
    df_ranking["sales_per_flow_percentile"] <= 5
)

# 유동인구 하·상위 10%
df_ranking["low_flow_flag"] = (
    df_ranking["flow_percentile"] <= 10
)

df_ranking["high_flow_flag"] = (
    df_ranking["flow_percentile"] >= 90
)

# 매출 상·하위 25%
df_ranking["high_sales_flag"] = (
    df_ranking["sales_percentile"] >= 75
)

df_ranking["low_sales_flag"] = (
    df_ranking["sales_percentile"] <= 25
)

## 8-7. 성과 발생 패턴 분류

In [ ]:
conditions = [
    # 고성과 유형
    (
        df_ranking["high_sales_per_flow_flag"]
        & df_ranking["high_sales_flag"]
        & df_ranking["low_flow_flag"]
    ),
    (
        df_ranking["high_sales_per_flow_flag"]
        & df_ranking["high_sales_flag"]
    ),
    (
        df_ranking["high_sales_per_flow_flag"]
        & df_ranking["low_flow_flag"]
    ),
    df_ranking["high_sales_per_flow_flag"],

    # 저성과 유형
    (
        df_ranking["low_sales_per_flow_flag"]
        & df_ranking["low_sales_flag"]
        & df_ranking["high_flow_flag"]
    ),
    (
        df_ranking["low_sales_per_flow_flag"]
        & df_ranking["low_sales_flag"]
    ),
    (
        df_ranking["low_sales_per_flow_flag"]
        & df_ranking["high_flow_flag"]
    ),
    df_ranking["low_sales_per_flow_flag"]
]

choices = [
    "high_sales_and_low_flow",
    "high_sales_pattern",
    "low_flow_pattern",
    "other_high_performance",

    "low_sales_and_high_flow",
    "low_sales_pattern",
    "high_flow_pattern",
    "other_low_performance"
]

df_ranking["performance_pattern"] = np.select(
    conditions,
    choices,
    default="normal"
)

pattern_summary = (
    df_ranking["performance_pattern"]
    .value_counts()
    .rename_axis("performance_pattern")
    .reset_index(name="trade_area_count")
)

display(pattern_summary)

## 8-8. 작은 상업 기반 주의 플래그

In [ ]:
df_ranking["small_commercial_base_flag"] = (
    (df_ranking["avg_sales_industry_count"] < 5)
    | (df_ranking["avg_store_count"] < 5)
)

df_ranking["low_transaction_flag"] = (
    df_ranking["total_sales_count"] < 1000
)

df_ranking["ranking_caution_flag"] = (
    df_ranking["small_commercial_base_flag"]
    | df_ranking["low_transaction_flag"]
)

caution_summary = (
    df_ranking["ranking_caution_flag"]
    .value_counts(dropna=False)
    .rename_axis("ranking_caution_flag")
    .reset_index(name="trade_area_count")
)

display(caution_summary)

## 8-9. 상권 유형 내부 지표 계산

In [ ]:
# 같은 유형의 상권 수
df_ranking["type_trade_area_count"] = (
    df_ranking
    .groupby("trade_area_type_name")["trade_area_code"]
    .transform("nunique")
)

# 같은 유형의 sales_per_flow 중앙값
df_ranking["type_median_sales_per_flow"] = (
    df_ranking
    .groupby("trade_area_type_name")["sales_per_flow"]
    .transform("median")
)

# 유형 중앙값 대비 상대성과
df_ranking["sales_per_flow_vs_type_median"] = safe_divide(
    df_ranking["sales_per_flow"],
    df_ranking["type_median_sales_per_flow"]
)

# 같은 유형 내부 순위
df_ranking["type_sales_per_flow_rank"] = (
    df_ranking
    .groupby("trade_area_type_name")["sales_per_flow"]
    .rank(method="min", ascending=False)
    .astype("Int64")
)

# 같은 유형 내부 백분위
df_ranking["type_sales_per_flow_percentile"] = (
    df_ranking
    .groupby("trade_area_type_name")["sales_per_flow"]
    .rank(pct=True)
    .mul(100)
)

df_ranking["type_rank_label"] = (
    df_ranking["type_sales_per_flow_rank"].astype("string")
    + " / "
    + df_ranking["type_trade_area_count"].astype("string")
)

## 8-10. 유형별 요약표

In [ ]:
type_summary = (
    df_ranking
    .groupby("trade_area_type_name", as_index=False)
    .agg(
        trade_area_count=("trade_area_code", "nunique"),

        total_sales_sum=("total_sales", "sum"),
        total_floating_population_sum=(
            "total_floating_population",
            "sum"
        ),

        median_total_sales=("total_sales", "median"),
        median_floating_population=(
            "total_floating_population",
            "median"
        ),

        mean_sales_per_flow=("sales_per_flow", "mean"),
        median_sales_per_flow=("sales_per_flow", "median"),

        median_transactions_per_flow=(
            "transactions_per_flow",
            "median"
        ),
        median_transaction_value=(
            "average_transaction_value",
            "median"
        )
    )
)

type_summary["weighted_sales_per_flow"] = safe_divide(
    type_summary["total_sales_sum"],
    type_summary["total_floating_population_sum"]
)

type_summary = type_summary.sort_values(
    "median_sales_per_flow",
    ascending=False
).reset_index(drop=True)

display(type_summary)

## 8-11. 서울 전체 Top 5·Bottom 5

In [ ]:
top5 = (
    df_ranking
    .nlargest(5, "sales_per_flow")
    .copy()
)

bottom5 = (
    df_ranking
    .nsmallest(5, "sales_per_flow")
    .copy()
)

ranking_columns = [
    "sales_per_flow_rank",
    "trade_area_code",
    "trade_area_name",
    "trade_area_type_name",
    "district_name",

    "total_sales",
    "sales_percentile",

    "total_floating_population",
    "flow_percentile",

    "sales_per_flow",
    "transactions_per_flow",
    "average_transaction_value",

    "type_rank_label",
    "sales_per_flow_vs_type_median",

    "performance_pattern",
    "low_flow_flag",
    "ranking_caution_flag"
]

print("서울 전체 Top 5")
display(top5[ranking_columns].reset_index(drop=True))

print("서울 전체 Bottom 5")
display(bottom5[ranking_columns].reset_index(drop=True))

## 유형별 Top 3·Bottom 3

In [ ]:
type_top3 = (
    df_ranking
    .sort_values(
        ["trade_area_type_name", "sales_per_flow"],
        ascending=[True, False]
    )
    .groupby("trade_area_type_name")
    .head(3)
    .reset_index(drop=True)
)

type_bottom3 = (
    df_ranking
    .sort_values(
        ["trade_area_type_name", "sales_per_flow"],
        ascending=[True, True]
    )
    .groupby("trade_area_type_name")
    .head(3)
    .reset_index(drop=True)
)

type_ranking_columns = [
    "trade_area_type_name",
    "type_rank_label",
    "trade_area_name",
    "district_name",
    "total_sales",
    "total_floating_population",
    "sales_per_flow",
    "transactions_per_flow",
    "average_transaction_value",
    "flow_percentile",
    "performance_pattern",
    "ranking_caution_flag"
]

print("상권 유형별 Top 3")
display(type_top3[type_ranking_columns])

print("상권 유형별 Bottom 3")
display(type_bottom3[type_ranking_columns])

In [ ]:
df_ranking["few_store_flag"] = (
    df_ranking["avg_store_count"] < 5
)

df_ranking["narrow_industry_flag"] = (
    df_ranking["avg_sales_industry_count"] < 5
)

df_ranking["low_transaction_flag"] = (
    df_ranking["total_sales_count"] < 1000
)

df_ranking["ranking_caution_flag"] = (
    df_ranking["few_store_flag"]
    | df_ranking["narrow_industry_flag"]
    | df_ranking["low_transaction_flag"]
)

In [ ]:
selected_codes = pd.concat(
    [
        type_top3[["trade_area_code"]],
        type_bottom3[["trade_area_code"]]
    ]
)["trade_area_code"].unique()

caution_check_columns = [
    "trade_area_type_name",
    "trade_area_name",
    "total_sales",
    "total_sales_count",
    "avg_sales_industry_count",
    "avg_store_count",
    "few_store_flag",
    "narrow_industry_flag",
    "low_transaction_flag",
    "ranking_caution_flag"
]

caution_check = (
    df_ranking[
        df_ranking["trade_area_code"].isin(selected_codes)
    ]
    .sort_values(
        ["trade_area_type_name", "sales_per_flow"],
        ascending=[True, False]
    )
)

display(caution_check[caution_check_columns])

# 9. Top 5·Bottom 5 사례 비교

## 9-1. Top 5·Bottom 5 다시 생성 및 결합

In [ ]:
top5 = (
    df_ranking
    .nlargest(5, "sales_per_flow")
    .copy()
)

bottom5 = (
    df_ranking
    .nsmallest(5, "sales_per_flow")
    .copy()
)

top5["performance_group"] = "Top 5"
bottom5["performance_group"] = "Bottom 5"

df_top_bottom = pd.concat(
    [top5, bottom5],
    ignore_index=True
)

df_top_bottom["performance_group"] = pd.Categorical(
    df_top_bottom["performance_group"],
    categories=["Top 5", "Bottom 5"],
    ordered=True
)

print("비교 대상 상권 수:", len(df_top_bottom))

## 9-2. 개별 상권 확인

In [ ]:
detail_columns = [
    "performance_group",
    "sales_per_flow_rank",
    "trade_area_code",
    "trade_area_name",
    "trade_area_type_name",
    "district_name",

    "total_sales",
    "sales_percentile",

    "total_floating_population",
    "flow_percentile",

    "sales_per_flow",
    "transactions_per_flow",
    "average_transaction_value",

    "type_rank_label",
    "sales_per_flow_vs_type_median",

    "performance_pattern",
    "low_flow_flag",
    "ranking_caution_flag"
]

top_bottom_detail = (
    df_top_bottom[detail_columns]
    .sort_values(
        ["performance_group", "sales_per_flow"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

display(top_bottom_detail)

## 9-3. 상권 유형 구성 확인

In [ ]:
type_composition = pd.crosstab(
    index=df_top_bottom["performance_group"],
    columns=df_top_bottom["trade_area_type_name"],
    margins=True
)

display(type_composition)

## 9-4. 성과 패턴 및 주의 플래그 확인

In [ ]:
pattern_composition = pd.crosstab(
    index=df_top_bottom["performance_group"],
    columns=df_top_bottom["performance_pattern"],
    margins=True
)

display(pattern_composition)

In [ ]:
flag_summary = (
    df_top_bottom
    .groupby("performance_group", observed=True)
    .agg(
        trade_area_count=("trade_area_code", "count"),

        low_flow_count=(
            "low_flow_flag",
            "sum"
        ),

        few_store_count=(
            "few_store_flag",
            "sum"
        ),

        narrow_industry_count=(
            "narrow_industry_flag",
            "sum"
        ),

        low_transaction_count=(
            "low_transaction_flag",
            "sum"
        ),

        caution_count=(
            "ranking_caution_flag",
            "sum"
        )
    )
    .reset_index()
)

display(flag_summary)

## 9-5. 핵심 성과지표 중앙값 비교

In [ ]:
core_metrics = [
    "total_sales",
    "total_floating_population",
    "sales_per_flow",
    "total_sales_count",
    "transactions_per_flow",
    "average_transaction_value",
    "sales_per_flow_vs_type_median"
]

core_group_median = (
    df_top_bottom
    .groupby(
        "performance_group",
        observed=True
    )[core_metrics]
    .median()
    .T
)

core_group_median["top_to_bottom_ratio"] = (
    core_group_median["Top 5"]
    / core_group_median["Bottom 5"].replace(0, np.nan)
)

core_group_median = (
    core_group_median
    .reset_index()
    .rename(columns={"index": "metric"})
)

display(core_group_median)

## 9-6. 평균값도 보조적으로 확인

In [ ]:
core_group_mean = (
    df_top_bottom
    .groupby(
        "performance_group",
        observed=True
    )[core_metrics]
    .mean()
    .T
)

core_group_mean["top_to_bottom_ratio"] = (
    core_group_mean["Top 5"]
    / core_group_mean["Bottom 5"].replace(0, np.nan)
)

core_group_mean = (
    core_group_mean
    .reset_index()
    .rename(columns={"index": "metric"})
)

display(core_group_mean)

## 9-7. 거래 빈도와 객단가 비교

In [ ]:
decomposition_metrics = [
    "sales_per_flow",
    "transactions_per_flow",
    "average_transaction_value"
]

decomposition_comparison = (
    df_top_bottom
    .groupby(
        "performance_group",
        observed=True
    )[decomposition_metrics]
    .agg(["median", "mean", "min", "max"])
)

display(decomposition_comparison)

## 9-8. 개별 상권별 분해 결과

In [ ]:
decomposition_detail_columns = [
    "performance_group",
    "trade_area_name",
    "trade_area_type_name",
    "sales_per_flow",
    "transactions_per_flow",
    "average_transaction_value",
    "total_sales",
    "total_floating_population",
    "performance_pattern"
]

decomposition_detail = (
    df_top_bottom[decomposition_detail_columns]
    .sort_values(
        ["performance_group", "sales_per_flow"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

display(decomposition_detail)

# 10. Top 5·Bottom 5 구조적 특성 비교

## 10-1. 비교 함수 만들기

In [ ]:
def compare_group_metrics(data, metrics):
    grouped = data.groupby(
        "performance_group",
        observed=True
    )[metrics]

    median_table = grouped.median().T
    count_table = grouped.count().T

    result = pd.DataFrame({
        "metric": median_table.index,

        "top5_median": median_table["Top 5"],
        "bottom5_median": median_table["Bottom 5"],

        "top5_available_count": count_table["Top 5"],
        "bottom5_available_count": count_table["Bottom 5"]
    })

    result["top_to_bottom_ratio"] = (
        result["top5_median"]
        / result["bottom5_median"].replace(0, np.nan)
    )

    result["median_difference"] = (
        result["top5_median"]
        - result["bottom5_median"]
    )

    return result.reset_index(drop=True)

## 10-2. 점포 경쟁 비교

In [ ]:
store_metrics = [
    "avg_store_count",
    "avg_non_franchise_store_count",
    "avg_franchise_store_count",
    "stores_per_10k_flow",
    "stores_per_km2",
    "franchise_store_share",
    "sales_per_store"
]

store_comparison = compare_group_metrics(
    df_top_bottom,
    store_metrics
)

display(store_comparison)

In [ ]:
store_detail_columns = [
    "performance_group",
    "trade_area_name",
    "trade_area_type_name",
    "sales_per_flow",

    "avg_store_count",
    "avg_non_franchise_store_count",
    "avg_franchise_store_count",

    "stores_per_10k_flow",
    "stores_per_km2",
    "franchise_store_share",
    "sales_per_store",

    "few_store_flag",
    "narrow_industry_flag"
]

display(
    df_top_bottom[
        store_detail_columns
    ].sort_values(
        ["performance_group", "sales_per_flow"],
        ascending=[True, False]
    ).reset_index(drop=True)
)

## 10-3. 직장인구 비교

In [ ]:
worker_metrics = [
    "avg_worker_population",
    "worker_concentration",
    "workers_per_km2"
]

worker_comparison = compare_group_metrics(
    df_top_bottom,
    worker_metrics
)

display(worker_comparison)

In [ ]:
worker_detail_columns = [
    "performance_group",
    "trade_area_name",
    "trade_area_type_name",
    "sales_per_flow",

    "avg_worker_population",
    "worker_concentration",
    "workers_per_km2",
    "worker_population_quarter_count"
]

display(
    df_top_bottom[
        worker_detail_columns
    ].sort_values(
        ["performance_group", "sales_per_flow"],
        ascending=[True, False]
    ).reset_index(drop=True)
)

## 10-4. 집객시설 비교

In [ ]:
facility_metrics = [
    "avg_visitor_facility_count",
    "facilities_per_10k_flow",
    "facilities_per_km2",

    "avg_transport_facility_count",
    "avg_shopping_culture_facility_count",
    "avg_medical_facility_count",
    "avg_education_facility_count",
    "avg_public_finance_facility_count",
    "avg_accommodation_facility_count"
]

facility_comparison = compare_group_metrics(
    df_top_bottom,
    facility_metrics
)

display(facility_comparison)

In [ ]:
facility_detail_columns = [
    "performance_group",
    "trade_area_name",
    "trade_area_type_name",
    "sales_per_flow",

    "avg_visitor_facility_count",
    "facilities_per_10k_flow",
    "facilities_per_km2",

    "avg_transport_facility_count",
    "avg_shopping_culture_facility_count",
    "avg_medical_facility_count",
    "avg_education_facility_count",
    "avg_public_finance_facility_count",
    "avg_accommodation_facility_count",

    "facility_quarter_count"
]

display(
    df_top_bottom[
        facility_detail_columns
    ].sort_values(
        ["performance_group", "sales_per_flow"],
        ascending=[True, False]
    ).reset_index(drop=True)
)

## 10-5. 면적 및 밀도 비교

In [ ]:
area_metrics = [
    "trade_area_size_m2",
    "trade_area_size_km2",
    "sales_per_km2",
    "flow_per_km2",
    "stores_per_km2",
    "workers_per_km2",
    "facilities_per_km2"
]

area_comparison = compare_group_metrics(
    df_top_bottom,
    area_metrics
)

display(area_comparison)

In [ ]:
area_detail_columns = [
    "performance_group",
    "trade_area_name",
    "trade_area_type_name",
    "sales_per_flow",

    "trade_area_size_m2",
    "trade_area_size_km2",

    "sales_per_km2",
    "flow_per_km2",
    "stores_per_km2",
    "workers_per_km2",
    "facilities_per_km2"
]

display(
    df_top_bottom[
        area_detail_columns
    ].sort_values(
        ["performance_group", "sales_per_flow"],
        ascending=[True, False]
    ).reset_index(drop=True)
)

## 10-6. 주말·저녁 시간대 비교

In [ ]:
time_metrics = [
    "weekend_sales_share",
    "weekend_flow_share",
    "evening_sales_share",
    "evening_flow_share"
]

time_comparison = compare_group_metrics(
    df_top_bottom,
    time_metrics
)

display(time_comparison)

In [ ]:
time_detail_columns = [
    "performance_group",
    "trade_area_name",
    "trade_area_type_name",
    "sales_per_flow",

    "weekend_sales_share",
    "weekend_flow_share",
    "evening_sales_share",
    "evening_flow_share"
]

display(
    df_top_bottom[
        time_detail_columns
    ].sort_values(
        ["performance_group", "sales_per_flow"],
        ascending=[True, False]
    ).reset_index(drop=True)
)

## 10-7. 핵심 비교지표 한 표로 정리

In [ ]:
final_comparison_metrics = [
    # 성과 분해
    "sales_per_flow",
    "transactions_per_flow",
    "average_transaction_value",

    # 경쟁
    "avg_store_count",
    "stores_per_10k_flow",
    "sales_per_store",

    # 직장인구
    "avg_worker_population",
    "worker_concentration",

    # 집객시설
    "avg_visitor_facility_count",
    "facilities_per_10k_flow",
    "facilities_per_km2",

    # 면적과 밀도
    "trade_area_size_m2",
    "sales_per_km2",
    "flow_per_km2",

    # 시간대
    "weekend_sales_share",
    "evening_sales_share"
]

final_group_comparison = compare_group_metrics(
    df_top_bottom,
    final_comparison_metrics
)

display(final_group_comparison)

## 10-8. 전체 상권에서 집객시설과 매출의 연관성 확인

In [ ]:
association_columns = [
    "total_sales",
    "sales_per_flow",

    "avg_store_count",
    "stores_per_10k_flow",

    "avg_worker_population",
    "worker_concentration",

    "avg_visitor_facility_count",
    "facilities_per_10k_flow",
    "facilities_per_km2",

    "trade_area_size_m2"
]

association_data = df_ranking[
    association_columns
].copy()

spearman_matrix = association_data.corr(
    method="spearman"
)

association_factors = [
    "avg_store_count",
    "stores_per_10k_flow",
    "avg_worker_population",
    "worker_concentration",
    "avg_visitor_facility_count",
    "facilities_per_10k_flow",
    "facilities_per_km2",
    "trade_area_size_m2"
]

association_result = (
    spearman_matrix
    .loc[
        association_factors,
        ["total_sales", "sales_per_flow"]
    ]
    .reset_index()
    .rename(columns={"index": "factor"})
)

display(association_result)

| 지표 | Top 5 중앙값 | Bottom 5 중앙값 | 차이 |
|---|---:|---:|---:|
| 점포 수 | 167.75 | 4 | 약 42배 |
| 점포당 매출 | 약 24.4억 | 약 316만 | 약 774배 |
| 측정 유동인구 대비 거래건수 | 84.04 | 0.00029 | 매우 큰 차이 |
| 평균 결제금액 | 약 2.79만 | 약 1.75만 | 약 1.6배 |
| 집객시설 수 | 15 | 7 | 약 2.1배 |
| 직장인구 | 297 | 454 | Top 5가 더 작음 |

Top 5와 Bottom 5의 차이는 대형 시장·도매 상권과 소규모 골목상권의 구조적 차이를 강하게 반영한다. 
특히 측정 유동인구 대비 거래건수와 점포당 매출의 격차가 크지만, 이 지표를 실제 구매 전환율로 해석해서는 안 된다.

## 10-9. 상위 10%만 따로 확인 / 관광특구 제외

In [ ]:
df_type_validation = df_ranking[
    df_ranking["type_trade_area_count"] >= 20
].copy()

df_type_validation["within_type_group"] = np.select(
    [
        df_type_validation[
            "type_sales_per_flow_percentile"
        ] >= 90,

        df_type_validation[
            "type_sales_per_flow_percentile"
        ] <= 10
    ],
    [
        "Type Top 10%",
        "Type Bottom 10%"
    ],
    default="Middle 80%"
)

df_type_extremes = df_type_validation[
    df_type_validation["within_type_group"].isin(
        ["Type Top 10%", "Type Bottom 10%"]
    )
].copy()

In [ ]:
display(
    pd.crosstab(
        df_type_extremes["within_type_group"],
        df_type_extremes["trade_area_type_name"],
        margins=True
    )
)

In [ ]:
validation_metrics = [
    "transactions_per_flow",
    "average_transaction_value",

    "avg_store_count",
    "sales_per_store",
    "stores_per_10k_flow",

    "avg_worker_population",
    "worker_concentration",

    "avg_visitor_facility_count",
    "facilities_per_km2",

    "trade_area_size_m2",
    "weekend_sales_share",
    "evening_sales_share"
]

within_type_comparison = (
    df_type_extremes
    .groupby("within_type_group")[validation_metrics]
    .median()
    .T
)

within_type_comparison["top_to_bottom_ratio"] = (
    within_type_comparison["Type Top 10%"]
    / within_type_comparison[
        "Type Bottom 10%"
    ].replace(0, np.nan)
)

within_type_comparison = (
    within_type_comparison
    .reset_index()
    .rename(columns={"index": "metric"})
)

display(within_type_comparison)

| 지표 | 하위 10% 중앙값 | 상위 10% 중앙값 | 상위/하위 |
|---|---:|---:|---:|
| 거래 빈도 지표 | 0.015 | 1.215 | 약 80배 |
| 점포 수 | 12개 | 101.6개 | 약 8.5배 |
| 점포당 매출 | 약 5,116만 | 약 3억 6,756만 | 약 7.2배 |
| 직장인구 | 222명 | 835.5명 | 약 3.8배 |
| 집객시설 | 6개 | 14개 | 약 2.3배 |
| 시설 밀도 | 129.9개/㎢ | 214.7개/㎢ | 약 1.7배 |
| 평균 결제금액 | 약 24,863원 | 약 32,459원 | 약 1.3배 |
| 상권 면적 | 약 45,816㎡ | 약 52,932㎡ | 약 1.2배 |

동일한 상권 유형 내부에서 비교한 결과, 유동인구 대비 매출 성과 상위 상권은 하위 상권보다 점포 수, 점포당 매출, 직장인구 및 집객시설 수가 모두 높았다. 
이 가운데 가장 큰 차이는 점포 규모와 거래 발생 정도에서 나타났으며, 직장인구와 집객시설은 성과를 보조적으로 설명하는 특성으로 확인되었다. 
평균 결제금액과 상권 면적의 차이는 상대적으로 작았다.

# 11. 분기별 성과 순위와 안정성 검증

## 11-1. 분기별 성과지표 계산

In [ ]:
df_quarterly_check = df_quarterly[
    df_quarterly["total_sales"].notna()
    & (df_quarterly["total_sales"] > 0)
    & df_quarterly["total_sales_count"].notna()
    & (df_quarterly["total_sales_count"] > 0)
    & df_quarterly["total_floating_population"].notna()
    & (df_quarterly["total_floating_population"] > 0)
].copy()

df_quarterly_check["quarter_sales_per_flow"] = safe_divide(
    df_quarterly_check["total_sales"],
    df_quarterly_check["total_floating_population"]
)

df_quarterly_check["quarter_transactions_per_flow"] = safe_divide(
    df_quarterly_check["total_sales_count"],
    df_quarterly_check["total_floating_population"]
)

df_quarterly_check["quarter_average_transaction_value"] = (
    safe_divide(
        df_quarterly_check["total_sales"],
        df_quarterly_check["total_sales_count"]
    )
)

## 11-2. 분기별 서울 전체 순위 계산

In [ ]:
df_quarterly_check["quarter_sales_per_flow_rank"] = (
    df_quarterly_check
    .groupby("quarter_code")["quarter_sales_per_flow"]
    .rank(method="min", ascending=False)
    .astype("Int64")
)

df_quarterly_check["quarter_sales_per_flow_percentile"] = (
    df_quarterly_check
    .groupby("quarter_code")["quarter_sales_per_flow"]
    .rank(pct=True)
    .mul(100)
)

## 11-3. Top 5·Bottom 5와 강남 마이스 선택

In [ ]:
selected_case_codes = pd.unique(
    pd.concat(
        [
            top5["trade_area_code"],
            bottom5["trade_area_code"],
            pd.Series([MICE_TRADE_AREA_CODE])  # 강남 마이스
        ],
        ignore_index=True
    )
)

group_map = (
    df_top_bottom
    .set_index("trade_area_code")["performance_group"]
    .to_dict()
)

quarterly_cases = df_quarterly_check[
    df_quarterly_check["trade_area_code"].isin(
        selected_case_codes
    )
].copy()

quarterly_cases["analysis_group"] = (
    quarterly_cases["trade_area_code"]
    .map(group_map)
    .fillna("Special Case")
)

quarterly_cases.loc[
    quarterly_cases["trade_area_code"] == MICE_TRADE_AREA_CODE,
    "analysis_group"
] = "Gangnam MICE"

## 11-4. 분기별 상세 결과 확인

In [ ]:
quarterly_case_columns = [
    "analysis_group",
    "quarter_code",
    "trade_area_code",
    "trade_area_name",
    "trade_area_type_name",

    "total_sales",
    "total_floating_population",
    "quarter_sales_per_flow",

    "total_sales_count",
    "quarter_transactions_per_flow",
    "quarter_average_transaction_value",

    "quarter_sales_per_flow_rank",
    "quarter_sales_per_flow_percentile"
]

display(
    quarterly_cases[
        quarterly_case_columns
    ].sort_values(
        [
            "analysis_group",
            "trade_area_name",
            "quarter_code"
        ]
    ).reset_index(drop=True)
)

## 11-5. 분기별 안정성 요약

In [ ]:
stability_summary = (
    quarterly_cases
    .groupby(
        [
            "trade_area_code",
            "trade_area_name",
            "trade_area_type_name",
            "analysis_group"
        ],
        as_index=False
    )
    .agg(
        quarter_count=("quarter_code", "nunique"),

        mean_quarter_sales_per_flow=(
            "quarter_sales_per_flow",
            "mean"
        ),
        median_quarter_sales_per_flow=(
            "quarter_sales_per_flow",
            "median"
        ),
        min_quarter_sales_per_flow=(
            "quarter_sales_per_flow",
            "min"
        ),
        max_quarter_sales_per_flow=(
            "quarter_sales_per_flow",
            "max"
        ),
        std_quarter_sales_per_flow=(
            "quarter_sales_per_flow",
            "std"
        ),

        best_quarter_rank=(
            "quarter_sales_per_flow_rank",
            "min"
        ),
        worst_quarter_rank=(
            "quarter_sales_per_flow_rank",
            "max"
        ),

        min_quarter_percentile=(
            "quarter_sales_per_flow_percentile",
            "min"
        ),
        max_quarter_percentile=(
            "quarter_sales_per_flow_percentile",
            "max"
        )
    )
)

stability_summary["sales_per_flow_cv"] = safe_divide(
    stability_summary["std_quarter_sales_per_flow"],
    stability_summary["mean_quarter_sales_per_flow"]
)

In [ ]:
annual_reference = df_ranking[
    [
        "trade_area_code",
        "sales_per_flow_rank",
        "sales_per_flow",
        "performance_pattern"
    ]
].copy()

stability_summary = stability_summary.merge(
    annual_reference,
    on="trade_area_code",
    how="left"
)

stability_summary = stability_summary.sort_values(
    "sales_per_flow_rank"
).reset_index(drop=True)

display(stability_summary)

## Top 5·Bottom 5의 분기별 특징

Top 5는 모두 시장·도매시장 성격을 보였으며 다음 특징이 공통적으로 나타났다.

- 총매출이 높음
- 측정 유동인구가 상대적으로 적음
- 측정 유동인구 대비 거래건수가 매우 높음
- 4개 분기 동안 계속 상위권을 유지함

따라서 이번 분석의 Top 5는 일반적인 의미에서 유동인구가 실제 구매로 잘 전환되는 상권이라기보다, 보행 유동인구만으로 매출을 설명하기 어려운 시장·도매형 특수 상권으로 해석하는 것이 적절하다. 매출과 유동인구의 포착 범위 차이도 높은 비율에 영향을 줬을 가능성이 있다.

Bottom 5는 대부분 점포가 3~4개 정도이고 연간 거래건수가 매우 적었으며, 4개 분기 동안 계속 최하위권을 유지했다. 이는 낮은 성과뿐 아니라 작은 상업 기반의 특성도 함께 반영한 결과다.

## 강남 마이스 관광특구

- 서울 전체 연간 9위
- 관광특구 내부 1위
- 유동인구 하위 10%에는 해당하지 않음
- 4개 분기 모두 상위권을 유지
- 분기별 변동이 작음

강남 마이스 관광특구는 유동인구가 극단적으로 적어서 특정 분기에만 높은 비율이 나온 상권이 아니라, 2025년 네 분기 모두 서울 상위 10위권의 유동인구 대비 매출성과를 유지한 구조적 고성과 상권이다. 이러한 안정성 때문에 심층 사례로 추가 분석하였다.

# 12. 관광특구 6개 상세 비교

## 12-1. 관광특구 전체 확인

In [ ]:
tourism_zones = df_ranking[
    df_ranking["trade_area_type_name"] == "관광특구"
].copy()

tourism_zones["tourism_rank"] = (
    tourism_zones["sales_per_flow"]
    .rank(method="min", ascending=False)
    .astype("Int64")
)

tourism_zones = tourism_zones.sort_values(
    "tourism_rank"
).reset_index(drop=True)

tourism_columns = [
    "tourism_rank",
    "trade_area_name",
    "district_name",

    "total_sales",
    "total_floating_population",
    "sales_per_flow",

    "transactions_per_flow",
    "average_transaction_value",

    "avg_store_count",
    "sales_per_store",

    "avg_worker_population",
    "worker_concentration",

    "avg_visitor_facility_count",
    "facilities_per_km2",

    "trade_area_size_m2",
    "sales_per_km2"
]

display(tourism_zones[tourism_columns])

## 12-2. 관광특구 중앙값 대비 강남 마이스

In [ ]:
tourism_comparison_metrics = [
    "total_sales",
    "total_floating_population",
    "sales_per_flow",
    "transactions_per_flow",
    "average_transaction_value",

    "avg_store_count",
    "sales_per_store",

    "avg_worker_population",
    "worker_concentration",

    "avg_visitor_facility_count",
    "facilities_per_km2",

    "trade_area_size_m2",
    "sales_per_km2"
]

tourism_median = (
    tourism_zones[tourism_comparison_metrics]
    .median()
)

mice_values = (
    tourism_zones.loc[
        tourism_zones["trade_area_code"] == MICE_TRADE_AREA_CODE,
        tourism_comparison_metrics
    ]
    .iloc[0]
)

mice_vs_tourism = pd.DataFrame({
    "metric": tourism_comparison_metrics,
    "gangnam_mice": mice_values.values,
    "tourism_median": tourism_median.values
})

mice_vs_tourism["mice_to_tourism_median_ratio"] = (
    mice_vs_tourism["gangnam_mice"]
    / mice_vs_tourism[
        "tourism_median"
    ].replace(0, np.nan)
)

display(mice_vs_tourism)

| 지표 | 관광특구 중앙값 대비 | 해석 |
|---|---:|---|
| 총매출 | 0.51배 | 절대 매출은 중앙값의 약 절반 |
| 유동인구 | 0.029배 | 관광특구 중앙값의 약 2.9% |
| `sales_per_flow` | 16.15배 | 측정 유동인구 대비 매출성과가 매우 높음 |
| 측정 유동인구 대비 거래건수 | 16.76배 | 성과 차이의 핵심 |
| 평균 결제금액 | 0.96배 | 객단가는 거의 동일 |
| 점포 수 | 0.29배 | 점포가 특별히 많지는 않음 |
| 점포당 매출 | 1.65배 | 점포 생산성은 비교적 높음 |
| 직장인구 | 1.37배 | 관광특구 중앙값보다 많음 |
| 집객시설 수 | 0.23배 | 시설 개수는 오히려 적음 |
| 상권 면적 | 0.38배 | 상대적으로 작은 상권 |
| 면적당 매출 | 1.33배 | 밀도는 높지만 16배 차이를 모두 설명하지는 못함 |

### 강남 마이스의 업무·상업 복합 특성

강남 마이스는 다른 관광특구보다 직장인구 집중도가 높은 업무·상업 복합형 관광특구다.

집객시설 수는 관광특구 중앙값의 23%, 시설 밀도는 중앙값의 71% 수준이다. 다만 강남 마이스에는 코엑스처럼 개별 규모와 집객력이 큰 복합시설이 있다. 현재 데이터는 시설의 개수만 제공하므로 시설 규모, 수용 인원, 방문객 수, 전시·행사 횟수 및 대형 복합시설 여부를 측정하지 못한다.

### 강남 마이스 결론

강남 마이스 관광특구는 2025년 모든 분기에서 서울 전체 유동인구 대비 매출성과 8~10위를 유지했다. 관광특구 중앙값과 비교하면 총매출은 약 절반이지만 측정 유동인구는 약 2.9%에 불과해 상대적 매출성과가 약 16배 높게 나타났다. 평균 결제금액은 관광특구 중앙값과 유사했으므로 이러한 차이는 고액 소비보다 측정 유동인구 대비 거래 발생 강도와 관련된 것으로 나타났다. 또한 직장인구 집중도가 높아 업무·상업 복합형 상권의 특징을 보였으며, 집객시설의 단순 개수만으로는 성과를 충분히 설명하지 못했다.

다음 장에서는 매출 데이터에 나타난 업종 수와 업종별 매출 비중을 이용해 상권의 업종 구조를 추가로 비교한다.

# 13. 업종 구성과 매출 집중도 분석

## 13-1. BigQuery에서 상권·업종별 연간 데이터 조회

In [ ]:
industry_query = """
WITH sales_quarter_industry AS (
    SELECT
        quarter_code,
        trade_area_code,

        ANY_VALUE(trade_area_name)
            AS trade_area_name,

        ANY_VALUE(trade_area_type_name)
            AS trade_area_type_name,

        industry_code,

        ANY_VALUE(industry_name)
            AS industry_name,

        SUM(sales_amount)
            AS sales_amount,

        SUM(sales_count)
            AS sales_count

    FROM `bonggeunj.seoul_commercial_dbt.stg_sales`

    WHERE quarter_code BETWEEN 20251 AND 20254

    GROUP BY
        quarter_code,
        trade_area_code,
        industry_code
),

store_quarter_industry AS (
    SELECT
        quarter_code,
        trade_area_code,
        industry_code,

        SUM(total_store_count)
            AS total_store_count,

        SUM(non_franchise_store_count)
            AS non_franchise_store_count,

        SUM(franchise_store_count)
            AS franchise_store_count

    FROM `bonggeunj.seoul_commercial_dbt.stg_store`

    WHERE quarter_code BETWEEN 20251 AND 20254

    GROUP BY
        quarter_code,
        trade_area_code,
        industry_code
)

SELECT
    s.trade_area_code,

    ANY_VALUE(s.trade_area_name)
        AS trade_area_name,

    ANY_VALUE(s.trade_area_type_name)
        AS trade_area_type_name,

    s.industry_code,

    ANY_VALUE(s.industry_name)
        AS industry_name,

    SUM(s.sales_amount)
        AS annual_industry_sales,

    SUM(s.sales_count)
        AS annual_industry_sales_count,

    COUNT(DISTINCT s.quarter_code)
        AS sales_quarter_count,

    AVG(st.total_store_count)
        AS avg_industry_store_count,

    AVG(st.non_franchise_store_count)
        AS avg_industry_non_franchise_store_count,

    AVG(st.franchise_store_count)
        AS avg_industry_franchise_store_count,

    COUNT(
        DISTINCT IF(
            st.total_store_count IS NOT NULL,
            s.quarter_code,
            NULL
        )
    ) AS store_quarter_count

FROM sales_quarter_industry AS s

LEFT JOIN store_quarter_industry AS st
    ON s.quarter_code = st.quarter_code
   AND s.trade_area_code = st.trade_area_code
   AND s.industry_code = st.industry_code

GROUP BY
    s.trade_area_code,
    s.industry_code

ORDER BY
    s.trade_area_code,
    annual_industry_sales DESC
"""

df_industry = client.query(
    industry_query,
    location="asia-northeast3"
).to_dataframe()

print("상권·업종 행 수:", len(df_industry))
print("상권 수:", df_industry["trade_area_code"].nunique())
print("업종 수:", df_industry["industry_code"].nunique())

display(df_industry.head())

## 13-2. 상권코드 자료형 통일

In [ ]:
def normalize_trade_area_code(series):
    return (
        pd.to_numeric(
            series,
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
    )


df_industry["trade_area_code"] = (
    normalize_trade_area_code(
        df_industry["trade_area_code"]
    )
)

## 13-3. 업종별 파생지표 계산

In [ ]:
# 상권별 연간 총매출
df_industry["trade_area_industry_sales_total"] = (
    df_industry
    .groupby("trade_area_code")[
        "annual_industry_sales"
    ]
    .transform("sum")
)

# 각 업종의 상권 내 매출 비중
df_industry["industry_sales_share"] = safe_divide(
    df_industry["annual_industry_sales"],
    df_industry["trade_area_industry_sales_total"]
)

# 업종별 평균 결제금액
df_industry["industry_average_transaction_value"] = (
    safe_divide(
        df_industry["annual_industry_sales"],
        df_industry["annual_industry_sales_count"]
    )
)

# 업종별 점포당 연간 매출
df_industry["industry_sales_per_store"] = safe_divide(
    df_industry["annual_industry_sales"],
    df_industry["avg_industry_store_count"]
)

# 상권 내부 업종 매출 순위
df_industry["industry_sales_rank"] = (
    df_industry
    .groupby("trade_area_code")[
        "annual_industry_sales"
    ]
    .rank(method="first", ascending=False)
    .astype("Int64")
)

display(
    df_industry[
        [
            "trade_area_code",
            "trade_area_name",
            "industry_code",
            "industry_name",
            "annual_industry_sales",
            "industry_sales_share",
            "industry_sales_rank",
            "avg_industry_store_count",
            "industry_sales_per_store"
        ]
    ].head(20)
)

## 13-4. 상권별 업종 집중도 요약

In [ ]:
industry_summary = (
    df_industry
    .groupby(
        "trade_area_code",
        as_index=False
    )
    .agg(
        annual_industry_sales_total=(
            "annual_industry_sales",
            "sum"
        ),

        annual_sales_industry_count=(
            "industry_code",
            "nunique"
        ),

        industry_sales_hhi=(
            "industry_sales_share",
            lambda x: (x ** 2).sum()
        )
    )
)

In [ ]:
industry_count = (
    industry_summary[
        "annual_sales_industry_count"
    ]
)

raw_hhi = industry_summary[
    "industry_sales_hhi"
]

industry_summary[
    "normalized_industry_sales_hhi"
] = np.where(
    industry_count <= 1,
    1.0,
    (
        raw_hhi
        - (1 / industry_count)
    )
    / (
        1
        - (1 / industry_count)
    )
)

industry_summary[
    "normalized_industry_sales_hhi"
] = (
    industry_summary[
        "normalized_industry_sales_hhi"
    ]
    .clip(0, 1)
)

In [ ]:
industry_summary[
    "industry_concentration_level"
] = np.select(
    [
        industry_summary["industry_sales_hhi"] <= 0.15,
        industry_summary["industry_sales_hhi"] <= 0.25
    ],
    [
        "diversified",
        "moderate"
    ],
    default="concentrated"
)

- `diversified`: 여러 업종에 매출이 분산된 상권
- `moderate`: 중간 수준으로 매출이 집중된 상권
- `concentrated`: 소수 업종에 매출이 집중된 상권

여기서 업종 수와 HHI는 상권에 존재하는 전체 업종이 아니라 `stg_sales`에 매출 행이 존재하는 업종을 기준으로 계산한다. 

HHI 0.15와 0.25는 본 분석에서 상권을 비교하기 위해 적용한 해석 기준이며, 업종 수 차이의 영향을 보완하기 위해 정규화 HHI도 함께 제공한다.

## 13-5. 매출 1위 업종 추출

In [ ]:
dominant_industry = (
    df_industry
    .sort_values(
        [
            "trade_area_code",
            "annual_industry_sales"
        ],
        ascending=[True, False]
    )
    .drop_duplicates(
        subset=["trade_area_code"],
        keep="first"
    )
    [
        [
            "trade_area_code",
            "industry_code",
            "industry_name",
            "annual_industry_sales",
            "annual_industry_sales_count",
            "industry_sales_share",
            "avg_industry_store_count",
            "industry_sales_per_store",
            "industry_average_transaction_value"
        ]
    ]
    .rename(
        columns={
            "industry_code":
                "dominant_industry_code",

            "industry_name":
                "dominant_industry_name",

            "annual_industry_sales":
                "dominant_industry_sales",

            "annual_industry_sales_count":
                "dominant_industry_sales_count",

            "industry_sales_share":
                "dominant_industry_sales_share",

            "avg_industry_store_count":
                "dominant_industry_avg_store_count",

            "industry_sales_per_store":
                "dominant_industry_sales_per_store",

            "industry_average_transaction_value":
                "dominant_industry_average_transaction_value"
        }
    )
)

## 13-6. 상위 3개 업종 매출 비중

In [ ]:
top3_industry_share = (
    df_industry[
        df_industry["industry_sales_rank"] <= 3
    ]
    .groupby(
        "trade_area_code",
        as_index=False
    )
    .agg(
        top3_industry_sales_share=(
            "industry_sales_share",
            "sum"
        )
    )
)

In [ ]:
industry_summary = (
    industry_summary
    .merge(
        dominant_industry,
        on="trade_area_code",
        how="left",
        validate="one_to_one"
    )
    .merge(
        top3_industry_share,
        on="trade_area_code",
        how="left",
        validate="one_to_one"
    )
)

print("업종 요약 상권 수:", len(industry_summary))
display(industry_summary.head())

## 13-7. 연간 매출 합계 검증

In [ ]:
sales_validation_base = df_ranking[
    [
        "trade_area_code",
        "total_sales"
    ]
].copy()

sales_validation_base["trade_area_code"] = (
    normalize_trade_area_code(
        sales_validation_base["trade_area_code"]
    )
)

sales_validation = (
    sales_validation_base
    .merge(
        industry_summary[
            [
                "trade_area_code",
                "annual_industry_sales_total"
            ]
        ],
        on="trade_area_code",
        how="left",
        validate="one_to_one"
    )
)

sales_validation["sales_difference"] = (
    sales_validation["total_sales"]
    - sales_validation[
        "annual_industry_sales_total"
    ]
).abs()

sales_validation["sales_relative_difference"] = safe_divide(
    sales_validation["sales_difference"],
    sales_validation["total_sales"]
)

print(
    "매출 상대오차 최대값:",
    sales_validation[
        "sales_relative_difference"
    ].max()
)

print(
    "상대오차 0.001보다 큰 상권 수:",
    (
        sales_validation[
            "sales_relative_difference"
        ] > 0.001
    ).sum()
)

## 13-8. df_ranking과 결합해 Looker용 데이터 생성

In [ ]:
df_looker = df_ranking.copy()

df_looker["trade_area_code"] = (
    normalize_trade_area_code(
        df_looker["trade_area_code"]
    )
)

df_looker = df_looker.merge(
    industry_summary,
    on="trade_area_code",
    how="left",
    validate="one_to_one"
)

print("df_ranking 행 수:", len(df_ranking))
print("df_looker 행 수:", len(df_looker))

print(
    "업종 요약 결측 상권 수:",
    df_looker[
        "dominant_industry_name"
    ].isna().sum()
)

## 13-9. Top 5·Bottom 5·강남 마이스 비교

In [ ]:
top5_codes = normalize_trade_area_code(
    top5["trade_area_code"]
)

bottom5_codes = normalize_trade_area_code(
    bottom5["trade_area_code"]
)

mice_codes = df_looker.loc[
    df_looker["trade_area_name"]
    .str.contains("마이스", na=False),
    "trade_area_code"
]

case_codes = pd.concat(
    [
        top5_codes,
        bottom5_codes,
        mice_codes
    ],
    ignore_index=True
).drop_duplicates()

industry_case_summary = (
    df_looker[
        df_looker["trade_area_code"].isin(
            case_codes
        )
    ]
    [
        [
            "sales_per_flow_rank",
            "trade_area_code",
            "trade_area_name",
            "trade_area_type_name",

            "total_sales",
            "total_floating_population",
            "sales_per_flow",

            "annual_sales_industry_count",
            "dominant_industry_name",
            "dominant_industry_sales_share",
            "top3_industry_sales_share",

            "industry_sales_hhi",
            "normalized_industry_sales_hhi",
            "industry_concentration_level",

            "dominant_industry_avg_store_count",
            "dominant_industry_sales_per_store"
        ]
    ]
    .sort_values("sales_per_flow_rank")
    .reset_index(drop=True)
)

display(industry_case_summary)

## 13-10. 사례 상권의 매출 상위 5개 업종

In [ ]:
case_industry_top5 = (
    df_industry[
        df_industry["trade_area_code"].isin(
            case_codes
        )
        & (
            df_industry[
                "industry_sales_rank"
            ] <= 5
        )
    ]
    [
        [
            "trade_area_name",
            "trade_area_type_name",
            "industry_sales_rank",
            "industry_code",
            "industry_name",

            "annual_industry_sales",
            "industry_sales_share",
            "annual_industry_sales_count",

            "avg_industry_store_count",
            "industry_sales_per_store",
            "industry_average_transaction_value"
        ]
    ]
    .sort_values(
        [
            "trade_area_name",
            "industry_sales_rank"
        ]
    )
    .reset_index(drop=True)
)

display(case_industry_top5)

In [ ]:
df_looker["single_industry_flag"] = (
    df_looker["annual_sales_industry_count"] == 1
)

df_looker["industry_coverage_group"] = np.select(
    [
        df_looker["annual_sales_industry_count"] == 1,

        df_looker[
            "annual_sales_industry_count"
        ].between(2, 4),

        df_looker["annual_sales_industry_count"] >= 5
    ],
    [
        "single_industry",
        "limited_industries",
        "multiple_industries"
    ],
    default="unknown"
)

display(
    df_looker[
        "industry_coverage_group"
    ].value_counts(dropna=False)
)

In [ ]:
df_looker["comparable_scope_flag"] = (
    df_looker["annual_sales_industry_count"] >= 5
)

In [ ]:
df_comparable = df_looker[
    df_looker["comparable_scope_flag"]
].copy()

df_comparable["comparable_sales_per_flow_rank"] = (
    df_comparable["sales_per_flow"]
    .rank(method="min", ascending=False)
    .astype("Int64")
)

display(
    df_comparable[
        [
            "comparable_sales_per_flow_rank",
            "trade_area_name",
            "trade_area_type_name",
            "sales_per_flow",
            "annual_sales_industry_count",
            "dominant_industry_name",
            "dominant_industry_sales_share",
            "industry_sales_hhi"
        ]
    ]
    .sort_values("comparable_sales_per_flow_rank")
    .head(20)
)

In [ ]:
df_diversified = df_looker[
    df_looker["comparable_scope_flag"]
    & (
        df_looker["industry_sales_hhi"]
        <= 0.15
    )
].copy()

df_diversified["diversified_sales_per_flow_rank"] = (
    df_diversified["sales_per_flow"]
    .rank(method="min", ascending=False)
    .astype("Int64")
)

diversified_columns = [
    "diversified_sales_per_flow_rank",
    "trade_area_name",
    "trade_area_type_name",
    "district_name",

    "sales_per_flow",
    "total_sales",
    "total_floating_population",

    "annual_sales_industry_count",
    "dominant_industry_name",
    "dominant_industry_sales_share",
    "top3_industry_sales_share",
    "industry_sales_hhi",

    "transactions_per_flow",
    "average_transaction_value"
]

display(
    df_diversified[
        diversified_columns
    ]
    .sort_values(
        "diversified_sales_per_flow_rank"
    )
    .head(20)
    .reset_index(drop=True)
)

In [ ]:
# 다업종 분산형 상권 여부
df_looker["diversified_trade_area_flag"] = (
    (df_looker["annual_sales_industry_count"] >= 5)
    & (df_looker["industry_sales_hhi"] <= 0.15)
)

# 다업종 분산형 상권 내 매출/유동인구 순위
df_looker["diversified_sales_per_flow_rank"] = (
    df_looker
    .groupby("diversified_trade_area_flag")["sales_per_flow"]
    .rank(method="min", ascending=False)
)

# 분산형 상권이 아닌 행은 순위 제거
df_looker.loc[
    ~df_looker["diversified_trade_area_flag"],
    "diversified_sales_per_flow_rank"
] = pd.NA

df_looker["diversified_sales_per_flow_rank"] = (
    df_looker["diversified_sales_per_flow_rank"].astype("Int64")
)

In [ ]:
display(
    df_looker.loc[
        df_looker["diversified_trade_area_flag"],
        [
            "diversified_sales_per_flow_rank",
            "trade_area_name",
            "trade_area_type_name",
            "sales_per_flow",
            "transactions_per_flow",
            "average_transaction_value",
            "annual_sales_industry_count",
            "dominant_industry_name",
            "dominant_industry_sales_share",
            "top3_industry_sales_share",
            "industry_sales_hhi",
        ]
    ]
    .sort_values("diversified_sales_per_flow_rank")
    .head(20)
)

강남 마이스 관광특구는 2025년 매출 데이터에 나타난 업종이 5개 이상이고 업종 매출 HHI가 0.15 이하인 상권 중 측정 유동인구 대비 매출성과가 가장 높았다.

코엑스보다 평균 결제금액은 약 6% 높았지만 측정 유동인구 대비 거래건수는 약 73% 높았다. 

따라서 두 상권의 성과 차이는 평균 결제금액보다 데이터에서 관측된 거래 발생 강도와 더 관련된 것으로 나타났다. 

다만 이 지표는 실제 구매 전환율을 의미하지 않는다.

# 전체 결론

## Top 5 특징

- 가락시장·강서농산물도매시장·동서시장·경동시장·인왕시장처럼 도매·전통시장 성격을 보임
- 총매출이 크면서 측정 유동인구가 상대적으로 적어 `sales_per_flow`가 높음
- 특정 핵심 업종의 매출 비중이 높은 전문화 상권

전체 Top 5는 도매시장과 전통시장이 대부분이며, 측정 유동인구에 비해 총매출과 거래건수가 매우 높았다. 

또한 특정 핵심 업종에 매출이 집중된 전문화 상권이라는 공통점이 나타났다. 

다만 이는 실제 방문자의 구매 전환율이 높다는 의미가 아니며, 도매 거래와 매출·유동인구 데이터의 포착 범위 차이가 결과에 영향을 줬을 가능성이 있다.

## Bottom 5 특징

Bottom 5는 측정 유동인구 대비 매출과 거래건수가 매우 낮았다. 

다만 매출 데이터에 나타난 업종 수와 점포 수가 적어 작은 상업 기반의 영향을 크게 받으며, 데이터가 상권 전체를 충분히 대표하지 못할 가능성도 있으므로 순위를 그대로 일반화해서는 안 된다.

## 강남 마이스 해석

- 매출 데이터에 나타난 업종 30개
- 가장 큰 업종인 일반의류 비중 약 21.2%
- 상위 3개 업종 비중 약 44.0%
- HHI 0.093으로 업종별 매출이 비교적 분산됨
- 매출 업종 5개 이상, HHI 0.15 이하 조건을 충족한 상권 중 측정 유동인구 대비 매출성과 1위

전체 상위권은 특정 도소매 업종에 매출이 집중된 전문시장형 상권이 대부분이었다. 

반면 강남 마이스 관광특구는 매출이 여러 업종에 비교적 분산되어 있으면서도 다업종 분산형 상권 중 측정 유동인구 대비 매출성과가 가장 높았다.

# 14. 공간·입지 특성과 상권 성과 분석

상권의 매출성과는 상권 자체의 규모와 업종 구성뿐 아니라 주변 상권의 밀집도와 행정구역·입지 특성의 영향을 함께 받을 수 있다. 이 장에서는 좌표, 행정구역, 상권 유형, 면적, 인구, 시설, 점포, 매출 및 업종 구성을 결합해 공간적 패턴을 탐색한다.

이 분석은 공간적 연관성을 확인하는 탐색적 분석이며, 특정 위치가 매출을 직접 변화시킨다는 인과관계를 의미하지 않는다.

## 14-1. 공간분석 변수 정의 및 데이터 품질 확인

공간분석에는 다음 정보를 사용한다.

1. X·Y 좌표
2. 자치구
3. 행정동
4. 상권 유형
5. 상권 면적
6. 유동인구
7. 직장인구
8. 집객시설
9. 점포
10. 매출과 거래건수
11. 업종 구성

좌표값은 위·경도가 아닌 서울시 상권 데이터의 투영좌표 형태다. 아래 거리 분석에서는 좌표 단위가 미터라는 전제하에 직선거리를 계산한다. 최종 지도 작성이나 외부 공간데이터 결합 전에는 서울 열린데이터광장의 원본 메타데이터에서 정확한 좌표계를 확인해야 한다.

In [ ]:
spatial_columns = [
    "trade_area_code",
    "trade_area_name",
    "x_coordinate",
    "y_coordinate",
    "district_name",
    "administrative_dong_name",
    "trade_area_type_name",
    "trade_area_size_m2",
    "total_floating_population",
    "avg_worker_population",
    "avg_visitor_facility_count",
    "avg_store_count",
    "total_sales",
    "total_sales_count",
    "sales_per_flow",
    "transactions_per_flow",
    "average_transaction_value",
    "annual_sales_industry_count",
    "dominant_industry_sales_share",
    "top3_industry_sales_share",
    "industry_sales_hhi",
]

df_spatial = df_looker[spatial_columns].copy()

spatial_quality = pd.DataFrame({
    "missing_count": df_spatial.isna().sum(),
    "missing_rate_pct": df_spatial.isna().mean() * 100,
}).sort_values("missing_count", ascending=False)

coordinate_summary = df_spatial[
    ["x_coordinate", "y_coordinate"]
].describe()

duplicate_coordinate_count = df_spatial.duplicated(
    ["x_coordinate", "y_coordinate"],
    keep=False,
).sum()

print("공간분석 대상 상권 수:", len(df_spatial))
print("좌표 결측 상권 수:", df_spatial[["x_coordinate", "y_coordinate"]].isna().any(axis=1).sum())
print("동일 좌표를 공유하는 상권 행 수:", duplicate_coordinate_count)
display(spatial_quality)
display(coordinate_summary)

## 14-2. 자치구·행정동·상권 유형별 성과 비교

서울 전체 순위는 대형 시장이나 관광특구 같은 특수 상권의 영향을 크게 받는다. 따라서 자치구와 상권 유형별 중앙값을 확인하고, 각 상권이 같은 자치구·상권 유형 안에서 어느 수준인지 계산한다. 행정동은 포함된 상권이 적을 수 있으므로 상권 수를 함께 표시한다.

In [ ]:
district_summary = (
    df_spatial
    .groupby("district_name", as_index=False)
    .agg(
        trade_area_count=("trade_area_code", "nunique"),
        administrative_dong_count=("administrative_dong_name", "nunique"),
        total_sales=("total_sales", "sum"),
        total_sales_count=("total_sales_count", "sum"),
        total_floating_population=("total_floating_population", "sum"),
        median_sales_per_flow=("sales_per_flow", "median"),
        median_transactions_per_flow=("transactions_per_flow", "median"),
        median_store_count=("avg_store_count", "median"),
        median_worker_population=("avg_worker_population", "median"),
        median_facility_count=("avg_visitor_facility_count", "median"),
        median_industry_count=("annual_sales_industry_count", "median"),
    )
)

district_summary["weighted_sales_per_flow"] = safe_divide(
    district_summary["total_sales"],
    district_summary["total_floating_population"],
)

district_summary["weighted_transactions_per_flow"] = safe_divide(
    district_summary["total_sales_count"],
    district_summary["total_floating_population"],
)

display(
    district_summary.sort_values(
        "median_sales_per_flow",
        ascending=False,
    )
)

In [ ]:
dong_summary = (
    df_spatial
    .groupby(
        ["district_name", "administrative_dong_name"],
        as_index=False,
    )
    .agg(
        trade_area_count=("trade_area_code", "nunique"),
        type_count=("trade_area_type_name", "nunique"),
        total_sales=("total_sales", "sum"),
        total_floating_population=("total_floating_population", "sum"),
        median_sales_per_flow=("sales_per_flow", "median"),
        median_transactions_per_flow=("transactions_per_flow", "median"),
    )
)

dong_summary["weighted_sales_per_flow"] = safe_divide(
    dong_summary["total_sales"],
    dong_summary["total_floating_population"],
)

display(
    dong_summary[
        dong_summary["trade_area_count"] >= 5
    ]
    .sort_values("median_sales_per_flow", ascending=False)
    .head(30)
)

In [ ]:
district_type_group = df_spatial.groupby(
    ["district_name", "trade_area_type_name"],
    observed=True,
)

df_spatial["district_type_trade_area_count"] = (
    district_type_group["trade_area_code"].transform("nunique")
)

df_spatial["district_type_median_sales_per_flow"] = (
    district_type_group["sales_per_flow"].transform("median")
)

df_spatial["sales_per_flow_vs_district_type_median"] = safe_divide(
    df_spatial["sales_per_flow"],
    df_spatial["district_type_median_sales_per_flow"],
)

df_spatial["district_type_sales_per_flow_percentile"] = (
    district_type_group["sales_per_flow"]
    .rank(pct=True)
    .mul(100)
)

display(
    df_spatial[
        df_spatial["district_type_trade_area_count"] >= 10
    ][
        [
            "trade_area_name",
            "district_name",
            "administrative_dong_name",
            "trade_area_type_name",
            "district_type_trade_area_count",
            "sales_per_flow",
            "sales_per_flow_vs_district_type_median",
            "district_type_sales_per_flow_percentile",
        ]
    ]
    .sort_values(
        "district_type_sales_per_flow_percentile",
        ascending=False,
    )
    .head(30)
)

## 14-3. 상권 자체 특성과 매출성과의 연관성

상권 면적, 유동인구, 직장인구, 집객시설, 점포, 거래건수 및 업종 구성이 총매출과 유동인구 대비 매출성과에 어떤 단조 관계를 보이는지 Spearman 상관계수로 확인한다. 상관계수는 인과효과가 아니며, 변수 간 규모 효과와 상권 유형 차이가 함께 반영될 수 있다.

In [ ]:
spatial_factor_columns = [
    "trade_area_size_m2",
    "total_floating_population",
    "avg_worker_population",
    "avg_visitor_facility_count",
    "avg_store_count",
    "total_sales_count",
    "annual_sales_industry_count",
    "dominant_industry_sales_share",
    "top3_industry_sales_share",
    "industry_sales_hhi",
]

spatial_association = (
    df_spatial[
        spatial_factor_columns
        + ["total_sales", "sales_per_flow", "transactions_per_flow"]
    ]
    .corr(method="spearman")
    .loc[
        spatial_factor_columns,
        ["total_sales", "sales_per_flow", "transactions_per_flow"],
    ]
    .reset_index()
    .rename(columns={"index": "factor"})
)

display(spatial_association)

## 14-4. 상권 간 거리 및 주변 상권 특성 계산

좌표가 존재하는 상권을 대상으로 중심점 간 직선거리를 계산한다. 반경 500m는 인접 도보권, 1km는 보다 넓은 주변 상권권역을 탐색하기 위한 기준으로 사용한다. 이는 실제 도로망 기반 보행거리나 상권 폴리곤 간 거리가 아니므로 탐색적 지표로 해석한다.

In [ ]:
coordinate_valid = df_spatial[
    df_spatial[["x_coordinate", "y_coordinate"]]
    .notna()
    .all(axis=1)
].copy()

coordinates = coordinate_valid[
    ["x_coordinate", "y_coordinate"]
].to_numpy(dtype=float)

coordinate_difference = (
    coordinates[:, np.newaxis, :]
    - coordinates[np.newaxis, :, :]
)

distance_matrix = np.sqrt(
    np.square(coordinate_difference).sum(axis=2)
)

np.fill_diagonal(distance_matrix, np.inf)

coordinate_valid["nearest_trade_area_distance"] = (
    distance_matrix.min(axis=1)
)

neighbor_metrics = [
    "total_sales",
    "total_sales_count",
    "total_floating_population",
    "avg_worker_population",
    "avg_visitor_facility_count",
    "avg_store_count",
    "annual_sales_industry_count",
    "industry_sales_hhi",
    "sales_per_flow",
]

for radius in [500, 1000]:
    neighbor_mask = distance_matrix <= radius
    coordinate_valid[f"neighbor_count_{radius}m"] = neighbor_mask.sum(axis=1)

    for metric in neighbor_metrics:
        values = coordinate_valid[metric].to_numpy(dtype=float)
        valid_values = ~np.isnan(values)

        neighbor_sum = neighbor_mask @ np.nan_to_num(values, nan=0.0)
        neighbor_valid_count = neighbor_mask @ valid_values.astype(int)

        coordinate_valid[f"neighbor_{metric}_mean_{radius}m"] = np.divide(
            neighbor_sum,
            neighbor_valid_count,
            out=np.full(len(coordinate_valid), np.nan),
            where=neighbor_valid_count > 0,
        )

neighbor_columns = [
    "trade_area_code",
    "nearest_trade_area_distance",
    "neighbor_count_500m",
    "neighbor_count_1000m",
] + [
    column
    for column in coordinate_valid.columns
    if column.startswith("neighbor_")
    and not column.startswith("neighbor_count_")
    and column.endswith(("500m", "1000m"))
]

df_spatial = df_spatial.merge(
    coordinate_valid[neighbor_columns],
    on="trade_area_code",
    how="left",
    validate="one_to_one",
)

display(
    df_spatial[
        [
            "trade_area_name",
            "district_name",
            "trade_area_type_name",
            "nearest_trade_area_distance",
            "neighbor_count_500m",
            "neighbor_count_1000m",
            "neighbor_total_sales_mean_500m",
            "neighbor_sales_per_flow_mean_500m",
        ]
    ]
    .sort_values("neighbor_count_500m", ascending=False)
    .head(30)
)

## 14-5. 주변 입지와 상권 성과의 연관성

주변 상권 수와 주변 상권의 인구·시설·점포·매출·업종 특성이 해당 상권의 성과와 어떤 관계를 보이는지 확인한다. 주변 평균에는 기준 상권 자체를 포함하지 않는다.

In [ ]:
neighborhood_factors = [
    "nearest_trade_area_distance",
    "neighbor_count_500m",
    "neighbor_count_1000m",
    "neighbor_total_sales_mean_500m",
    "neighbor_total_sales_count_mean_500m",
    "neighbor_total_floating_population_mean_500m",
    "neighbor_avg_worker_population_mean_500m",
    "neighbor_avg_visitor_facility_count_mean_500m",
    "neighbor_avg_store_count_mean_500m",
    "neighbor_annual_sales_industry_count_mean_500m",
    "neighbor_industry_sales_hhi_mean_500m",
    "neighbor_sales_per_flow_mean_500m",
]

neighborhood_association = (
    df_spatial[
        neighborhood_factors
        + ["total_sales", "sales_per_flow", "transactions_per_flow"]
    ]
    .corr(method="spearman")
    .loc[
        neighborhood_factors,
        ["total_sales", "sales_per_flow", "transactions_per_flow"],
    ]
    .reset_index()
    .rename(columns={"index": "neighborhood_factor"})
)

display(neighborhood_association)

## 14-6. 고성과·저성과 공간 패턴 탐색

1km 반경 이웃의 평균 성과와 서울 전체 중앙값을 기준으로 High–High, Low–Low, High–Low, Low–High 패턴을 구분한다. 이 분류는 유의성 검정을 포함한 Local Moran’s I가 아니라, 후속 공간분석 대상을 찾기 위한 기술적 분류다.

In [ ]:
overall_sales_per_flow_median = df_spatial["sales_per_flow"].median()

own_high = (
    df_spatial["sales_per_flow"]
    >= overall_sales_per_flow_median
)

neighbor_high = (
    df_spatial["neighbor_sales_per_flow_mean_1000m"]
    >= overall_sales_per_flow_median
)

has_neighbor = df_spatial["neighbor_count_1000m"] > 0

df_spatial["spatial_performance_pattern"] = np.select(
    [
        has_neighbor & own_high & neighbor_high,
        has_neighbor & ~own_high & ~neighbor_high,
        has_neighbor & own_high & ~neighbor_high,
        has_neighbor & ~own_high & neighbor_high,
    ],
    [
        "High-High",
        "Low-Low",
        "High-Low",
        "Low-High",
    ],
    default="No neighbor",
)

spatial_pattern_summary = (
    df_spatial["spatial_performance_pattern"]
    .value_counts(dropna=False)
    .rename_axis("spatial_performance_pattern")
    .reset_index(name="trade_area_count")
)

display(spatial_pattern_summary)

display(
    df_spatial[
        df_spatial["spatial_performance_pattern"].isin(
            ["High-Low", "Low-High"]
        )
    ][
        [
            "trade_area_name",
            "district_name",
            "administrative_dong_name",
            "trade_area_type_name",
            "sales_per_flow",
            "neighbor_sales_per_flow_mean_1000m",
            "neighbor_count_1000m",
            "spatial_performance_pattern",
            "annual_sales_industry_count",
            "industry_sales_hhi",
        ]
    ]
    .sort_values("sales_per_flow", ascending=False)
    .head(40)
)

## 14-7. 공간분석 결과 해석

### 데이터 품질과 공간 범위

- 공간분석 대상 1,558개 상권은 모두 X·Y 좌표를 보유했으며 동일 좌표를 공유하는 상권도 없었다.
- 직장인구는 6개 상권(0.39%), 집객시설은 53개 상권(3.40%)에서 결측이 확인되었다. 결측은 0으로 대체하지 않고 각 분석에서 제외하였다.
- 좌표는 서울 전역에 분포했지만 정확한 좌표계는 원본 메타데이터에서 추가 확인해야 한다. 현재 500m·1km 분석은 좌표 단위가 미터라는 전제의 중심점 간 직선거리다.

### 지역별 차이

- 자치구별 유동인구 대비 매출성과 중앙값은 중구 약 22,712, 강남구 약 18,311, 서초구 약 13,799 순으로 높았다.
- 하위권은 강북구 약 2,840, 성북구 약 3,640, 관악구 약 3,642로 나타나 자치구에 따른 성과 수준 차이가 확인되었다.
- 행정동별로는 중구 소공동의 중앙값이 약 72,634로 가장 높았으며, 강서구 가양1동 약 63,276, 중구 회현동 약 54,526이 뒤를 이었다. 다만 행정동별 상권 수와 상권 유형 구성이 다르므로 지역 순위만으로 입지효과를 단정할 수 없다.
- 같은 자치구·상권 유형 내부에서도 가락시장, 월드컵경기장역, 코엑스처럼 지역 중앙값을 크게 웃도는 상권이 존재했다. 이는 자치구 위치만으로 성과가 결정되지 않고 개별 상권의 업종과 거래 구조가 중요함을 보여준다.

### 상권 자체 특성과 성과

- 총매출과 가장 강한 양의 관계를 보인 변수는 매출건수(ρ=0.946), 점포 수(ρ=0.904), 매출 업종 수(ρ=0.858), 집객시설 수(ρ=0.733)였다.
- 유동인구 대비 매출성과는 매출건수(ρ=0.673), 점포 수(ρ=0.567), 매출 업종 수(ρ=0.498), 집객시설 수(ρ=0.410), 직장인구(ρ=0.387)와 양의 관계를 보였다.
- 측정 유동인구 대비 거래건수도 매출건수(ρ=0.782), 점포 수(ρ=0.613), 업종 수(ρ=0.560), 집객시설 수(ρ=0.474)와 비교적 강한 양의 관계를 보였다.
- 반대로 상위 3개 업종 매출 비중(ρ=-0.390)과 업종 HHI(ρ=-0.340)는 유동인구 대비 매출성과와 음의 관계를 보였다. 전체적으로는 특정 업종 의존도가 낮고 여러 업종과 점포를 보유한 상권이 더 높은 성과를 보이는 경향이 있었다. 다만 일부 도매·전통시장은 높은 업종 집중도와 높은 성과가 동시에 나타나는 예외 사례다.

### 주변 상권과 공간적 집적

- 500m 안에 상권이 가장 많이 모인 곳은 동대문·동묘·청량리 전통시장 권역으로, 최대 18개 상권이 확인되었다. 이는 시장과 상가가 인접해 있는 공간적 집적을 잘 보여준다.
- 그러나 반경 내 상권 수와 해당 상권의 유동인구 대비 매출성과 간 상관은 500m에서 ρ=0.029, 1km에서 ρ=-0.048로 매우 약했다. 단순히 주변 상권이 많다는 사실만으로 개별 상권의 성과가 높아지지는 않았다.
- 주변 상권 평균 성과와 해당 상권 성과의 관계도 ρ=0.157로 약한 수준이었다. 주변 직장인구 평균(ρ=0.130)과 주변 업종 HHI 평균(ρ=0.129) 역시 약한 양의 관계에 그쳤다.
- 따라서 현재 데이터에서는 광역적인 위치나 단순 상권 밀집도보다 해당 상권의 거래건수, 점포 규모, 업종 구성 같은 자체 특성이 성과와 더 밀접하게 연결된 것으로 나타났다.

### 공간 패턴 후보군

- 1km 이웃 평균과 서울 전체 중앙값을 이용한 기술적 분류 결과 High–High 702개, Low–High 689개, Low–Low 86개, High–Low 71개, 이웃 없음 10개로 나타났다.
- 마들역, 구로역, 수유전통시장 등 High–Low 상권은 주변보다 상대적으로 높은 성과를 보이는 독립적 고성과 후보군이다.
- 다만 이 분류는 이웃 평균이 일부 극단값의 영향을 받을 수 있고 통계적 유의성 검정을 포함하지 않는다. 따라서 정식 공간 군집으로 결론 내리기보다 후속 지도 검토와 Local Moran’s I 분석을 위한 후보군으로 사용한다.

### 종합 해석

분석 결과 위치에 따른 지역 차이와 시장 밀집 권역은 확인됐지만, 반경 내 단순 상권 수와 주변 평균 특성만으로 개별 상권의 매출성과를 충분히 설명하지는 못했다. 현재 단계에서는 상권의 위치 자체보다 점포 수, 거래건수, 업종 다양성 및 집객시설 같은 상권 내부 구조의 연관성이 더 크게 나타났다.

후속 분석에서는 정확한 좌표계 확인 후 지하철역 거리와 승하차 인원, 도로망 기반 보행거리, 토지이용, 업무시설 및 대형 복합시설 규모를 결합할 필요가 있다. 또한 상권 유형과 규모를 통제한 회귀모형이나 공간회귀를 적용해야 입지와 성과의 관계를 더 엄밀하게 평가할 수 있다. 모든 결과는 연관성을 보여줄 뿐 위치의 인과효과를 증명하지 않는다.

# 15. 분석 결과 BigQuery 업로드

> **주의:** 아래 셀은 분석 결과를 BigQuery에 적재하며, `WRITE_TRUNCATE` 설정으로 기존 `trade_area_annual_2025` 테이블을 덮어쓴다. 분석 검토만 수행할 때는 업로드 셀을 실행하지 않는다.

In [ ]:
df_upload = df_looker.copy()

# BigQuery에서 처리하기 어려운 무한대 값을 결측값으로 변경
df_upload = df_upload.replace([np.inf, -np.inf], np.nan)

# 컬럼명을 BigQuery에서 안전하게 사용할 수 있도록 정리
df_upload.columns = (
    df_upload.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9_]+", "_", regex=True)
    .str.strip("_")
)

# 상권코드는 숫자 계산 대상이 아니므로 문자열로 유지
df_upload["trade_area_code"] = (
    df_upload["trade_area_code"].astype("string")
)

# category 자료형이 있다면 문자열로 변환
for column in df_upload.select_dtypes(include=["category"]).columns:
    df_upload[column] = df_upload[column].astype("string")

print("업로드 행 수:", len(df_upload))
print("상권코드 고유 개수:", df_upload["trade_area_code"].nunique())
print("컬럼 수:", len(df_upload.columns))

In [ ]:
assert df_upload["trade_area_code"].notna().all(), \
    "상권코드에 결측값이 있습니다."

assert df_upload["trade_area_code"].nunique() == len(df_upload), \
    "상권코드가 중복되었습니다."

assert (df_upload["total_floating_population"] > 0).all(), \
    "유동인구가 0 이하인 행이 있습니다."

assert (df_upload["total_sales"] >= 0).all(), \
    "총매출이 음수인 행이 있습니다."

print("업로드 전 검증 통과")

In [ ]:
PROJECT_ID = "bonggeunj"
DATASET_NAME = "seoul_commercial_analysis"
LOCATION = "asia-northeast3"

DATASET_ID = f"{PROJECT_ID}.{DATASET_NAME}"

client = bigquery.Client(
    project=PROJECT_ID,
    location=LOCATION
)

dataset = bigquery.Dataset(DATASET_ID)
dataset.location = LOCATION

dataset = client.create_dataset(
    dataset,
    exists_ok=True
)

print("데이터셋 준비 완료:", DATASET_ID)
print("데이터셋 위치:", dataset.location)

In [ ]:
TABLE_NAME = "trade_area_annual_2025"
TABLE_ID = f"{DATASET_ID}.{TABLE_NAME}"

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

load_job = client.load_table_from_dataframe(
    dataframe=df_upload,
    destination=TABLE_ID,
    job_config=job_config
)

load_job.result()

uploaded_table = client.get_table(TABLE_ID)

print("업로드 완료")
print("테이블:", TABLE_ID)
print("행 수:", uploaded_table.num_rows)
print("컬럼 수:", len(uploaded_table.schema))

In [ ]:
validation_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT trade_area_code) AS distinct_trade_area_count,
    COUNTIF(trade_area_code IS NULL) AS missing_trade_area_code,
    COUNTIF(total_floating_population IS NULL) AS missing_floating_population,
    COUNTIF(total_floating_population <= 0) AS invalid_floating_population
FROM `{TABLE_ID}`
"""

df_upload_check = client.query(
    validation_query,
    location=LOCATION
).to_dataframe()

display(df_upload_check)